# 백혈구(WBC) 4종 분류 — 최종 통합본

**2조 프로젝트** / 데이터: Kaggle `paultimothymooney/blood-cells`
교재: **딥러닝의 정석 with 파이토치** (수업 1~5장)

---

## 이 노트북은 세 개의 노트북을 합친 것이다

| 출처 | 그 노트북의 강점 | 여기서 어떻게 썼나 |
|---|---|---|
| **3조 (원본분할)** | 원본을 먼저 나눠 **증강본이 분할을 넘지 못하게** 함. 라벨 품질 감사. 테스트를 마지막에 딱 한 번만 엶 | §2 라벨 정제, §5 분할 원칙, §16 테스트 1회 원칙 |
| **pro4_1** | **백혈구 ROI 자동 검출**, 클래스 가중치, 수작업 특징 베이스라인, 5겹 교차검증, **집중배율**(Grad-CAM 정량 채점), 염색 스트레스, 보류 임계값 τ | §4 ROI, §7 가중치, §9 M0, §17 CV, §18 집중배율, §19 염색, §20 τ |
| **전체과정(이전 버전)** | 수업 내용 매핑, 학습 루프 해부, 속도 프로브(3분 조건), 에폭 결정, 통계적 가설검정 | §1 §7 §8 §10 §21 |

**그리고 세 노트북이 모두 놓친 것을 이 노트북이 고친다.** → §3

---

## 결론부터 (실행 전 요약)

1. 이 데이터의 **12,444장 증강본은 366장 원본을 불린 것**이다. 그래서
   **증강본을 무작위로 나눠 만든 검증셋은 성능을 크게 부풀린다.**
   실제로 같은 모델이 무작위 검증 **0.9965**, 공식 TEST **0.8536** 을 냈다(이전 실행 기록).
2. 그 부풀린 검증셋으로 하이퍼파라미터를 고르면 **"증강을 하지 마라"** 같은 잘못된 결론이 나온다.
   외운 것을 다시 물어보는 시험이기 때문이다.
3. 그래서 이 노트북은 **모델 선택을 증강본이 아니라 '증강되지 않은 원본 354장'으로** 한다.
4. 배경(적혈구) 암기를 막기 위해 **백혈구 ROI만 잘라** 원본과 증강본의 화각을 맞춘다.
5. 최종 점수는 **한 번도 건드리지 않은 공식 TEST 2,487장**에서 딱 한 번 잰다.
6. 추가로 **누수가 원천적으로 불가능한 트랙**(원본 354장 5겹 교차검증)을 따로 돌려 대조한다.

---

## 목차

| § | 내용 | 에폭 |
|---|---|---|
| 0 | 실행 설정 | |
| 1 | 준비 — 환경·시드 | |
| 2 | 데이터와 라벨 품질 감사 | |
| **3** | **누수 진단 — 이 프로젝트의 핵심** | |
| 4 | 백혈구 ROI 자동 검출 | |
| 5 | 데이터셋 · 분할 · 증강 | |
| 6 | 모델 6종 | |
| 7 | 학습 · 평가 함수 | |
| 8 | 속도 프로브 (1 에폭 3분) | |
| 9 | 실험1 — 베이스라인 (수작업 특징 포함) | ○ |
| 10 | 실험2 — 에폭 수 결정 | ○ |
| 11 | 실험3 — 입력·증강 (ROI 효과) | ○ |
| 12 | 실험4 — 모델 비교 | ○ |
| 13 | 실험5 — 하이퍼파라미터 | ○ |
| 14 | 실험6 — 시드 반복 | ○ |
| 15 | 최종 학습 | ○ |
| 16 | **최종 평가 (공식 TEST, 딱 한 번)** | |
| 17 | 무누수 대조 트랙 (원본 5겹 교차검증) | ○ |
| 18 | Grad-CAM + 집중배율 | |
| 19 | 염색 스트레스 테스트 | |
| 20 | 보류 임계값 τ (운영 곡선) | |
| 21 | 가설검정 + 다중비교 보정 | |
| 22 | 최종 결론 |

## §0. 실행 설정

`RUN_LEVEL` 하나로 규모를 정한다. 중간에 멈춰도 `results/runs.csv` 에 누적되고
이미 돌린 실험은 자동으로 건너뛴다.

| 값 | 용도 | 대략 시간(GPU) |
|---|---|---|
| `'test'` | 코드가 끝까지 도는지 확인 | 15~25분 |
| `'quick'` | 발표 준비용 | 1.5~3시간 |
| `'full'` | 최종 제출용 | 5~8시간 |

In [ ]:
RUN_LEVEL = 'quick'          # 'test' | 'quick' | 'full'

PROFILE = {
    'test' : dict(screen_epochs=1, full_epochs=2,  subset=600,  seeds=[0,1,2],     cv_epochs=1,  cv_folds=3, image_size=160),
    'quick': dict(screen_epochs=4, full_epochs=15, subset=4000, seeds=[0,1,2],     cv_epochs=15, cv_folds=5, image_size=224),
    'full' : dict(screen_epochs=6, full_epochs=30, subset=None, seeds=[0,1,2,3,4], cv_epochs=40, cv_folds=5, image_size=224),
}[RUN_LEVEL]

SCREEN_EPOCHS = PROFILE['screen_epochs']   # 후보 비교용(짧게 — 지금 필요한 건 순위다)
FULL_EPOCHS   = PROFILE['full_epochs']     # 본 학습용
SUBSET        = PROFILE['subset']          # 스크리닝에 쓸 학습 장수 (None = 전체)
SEEDS         = PROFILE['seeds']
CV_EPOCHS     = PROFILE['cv_epochs']
CV_FOLDS      = PROFILE['cv_folds']
IMAGE_SIZE    = PROFILE['image_size']
BATCH_SIZE    = 32
NUM_WORKERS   = 4        # 윈도우 주피터에서 DataLoader 오류가 나면 0 으로 내린다
EPOCH_BUDGET  = 180      # 과제 조건: 1 에폭 최대 3분

print(f'실행 수준 {RUN_LEVEL}')
print(f'  스크리닝 {SCREEN_EPOCHS}에폭 / 본학습 {FULL_EPOCHS}에폭 / 교차검증 {CV_FOLDS}겹×{CV_EPOCHS}에폭')
print(f'  스크리닝 표본 {SUBSET or "전체"} / 시드 {SEEDS} / 입력 {IMAGE_SIZE}px')

---
# §1. 준비 — 환경과 재현성

> **📘 수업 1장**: `device` 확인 코드는 **모든 노트북의 첫 셀**이다. `cpu` 가 나오면 GPU 를 못 쓰는 상태다.
> GPU 연산은 **비동기**라서 시간을 재려면 `torch.cuda.synchronize()` 로 끝나기를 기다려야 한다.
>
> **📘 수업 2-1**: 시드를 고정해야 같은 결과가 재현된다. §14 에서 시드를 바꿔가며 반복하므로 특히 중요하다.

### 용어 정리 — 이 노트북에서 계속 나오는 말

| 용어 | 뜻 |
|---|---|
| **원본(original / parent)** | `dataset-master/JPEGImages` 의 640×480 사진 366장. 증강 전 |
| **증강본(child)** | `dataset2-master` 의 320×240 사진 12,444장. 원본을 회전·이동·확대해 불린 것 |
| **누수(leakage)** | 시험 문제의 답이 어떤 경로로든 학습에 새어 들어가는 것. 여기서는 *같은 원본의 형제 사진이 학습과 시험에 함께 들어가는 것* |
| **ROI** | Region Of Interest. 여기서는 백혈구 한 개가 들어 있는 정사각 영역 |
| **macro-F1** | 클래스별 F1 을 단순 평균한 값. 클래스가 불균형해도 소수 클래스를 무시하지 못하게 한다 |
| **집중배율** | Grad-CAM 이 세포 위에 얼마나 몰려 있는가. 1.0 = 아무 데나 본 것과 같음 |

In [ ]:
import os, sys, time, json, math, random, itertools, collections, shutil, glob, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage, stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, confusion_matrix,
                             classification_report)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device)
if device.type == 'cuda':
    print('GPU  :', torch.cuda.get_device_name(0))
else:
    print('⚠ CPU 로 도는 중이다. GPU 가 안 잡히면 3분 예산을 맞출 수 없다.')


def set_seed(seed=42):
    """수업 2-1. 재현성"""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True     # 입력 크기가 고정이라 켜면 빨라진다

set_seed(42)

In [ ]:
# 그래프 한글 폰트 (수업 1.ipynb 의 Malgun Gothic 지정과 같다)
import matplotlib
for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']:
    try:
        matplotlib.font_manager.findfont(f, fallback_to_default=False)
        plt.rcParams['font.family'] = f; break
    except Exception:
        continue
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.max_open_warning'] = 0
pd.set_option('display.max_columns', 40)
print('폰트:', plt.rcParams['font.family'])

# ---- 경로 --------------------------------------------------------------
AUG_DIR  = './dataset2-master/images'      # 증강본. TRAIN / TEST 의 부모
ORIG_DIR = './dataset-master'              # 원본 366장 + labels.csv
RESULT_DIR = './results'
for d in [RESULT_DIR, f'{RESULT_DIR}/ckpt', f'{RESULT_DIR}/preds']:
    os.makedirs(d, exist_ok=True)

print('증강본 폴더:', os.path.isdir(AUG_DIR), '| 원본 폴더:', os.path.isdir(ORIG_DIR))
assert os.path.isdir(AUG_DIR) and os.path.isdir(ORIG_DIR), \
    '경로를 확인하세요. 이 노트북은 dataset2-master 와 dataset-master 와 같은 폴더에 두고 실행합니다.'

---
# §2. 데이터와 라벨 품질 감사

> **📘 수업 5-1**: `ImageFolder` 는 폴더 이름을 클래스로 삼고 **알파벳 순으로 번호**를 매긴다.
> **`class_to_idx` 를 반드시 확인한다.** 이걸 확인 안 하면 혼동행렬과 CAM 해석이 통째로 뒤집힌다.

여기서는 원본 라벨(`labels.csv`)이 지저분해서 `ImageFolder` 를 쓸 수 없다.
**3조·pro4_1 이 한 라벨 감사를 그대로 가져온다.** 라벨을 추측하지 않고, 정답이 확실한 것만 쓴다.

In [ ]:
CLASS_NAMES = ['EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']   # 알파벳 순 = 라벨 0,1,2,3
KOR = {'EOSINOPHIL': '호산구', 'LYMPHOCYTE': '림프구',
       'MONOCYTE': '단핵구', 'NEUTROPHIL': '호중구'}
CLS2IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
print('class_to_idx :', CLS2IDX)

In [ ]:
def audit_labels(csv_path, img_dir):
    """원본 labels.csv 감사. 3조·pro4_1 의 처리 규칙을 합쳤다.

    - 'NEUTROPHIL, NEUTROPHIL' 처럼 같은 종류가 중복 기재 -> 하나로 복원해서 사용
    - 'NEUTROPHIL, EOSINOPHIL' 처럼 다른 종류 혼재      -> 이미지 단위 정답을 정할 수 없어 제외
    - BASOPHIL (호염기구)                              -> 4종 대상이 아니고 3장뿐이라 제외
    - 라벨 결측 / 파일 없음                             -> 제외
    """
    df = pd.read_csv(csv_path)
    rows, reasons, recovered = [], collections.Counter(), 0
    for img_id, cat in zip(df['Image'], df['Category']):
        fp = os.path.join(img_dir, f'BloodImage_{int(img_id):05d}.jpg')
        if not os.path.exists(fp):
            reasons['파일 없음'] += 1; continue
        if not isinstance(cat, str) or cat.strip() == '':
            reasons['라벨 결측'] += 1; continue
        parts = [p.strip() for p in cat.split(',')]
        if len(set(parts)) > 1:
            reasons['다중 라벨(혼재)'] += 1; continue
        if len(parts) > 1:
            recovered += 1                       # 같은 종류 중복 -> 복원
        c = parts[0]
        if c not in CLS2IDX:
            reasons[f'4종 밖({c})'] += 1; continue
        rows.append({'path': fp, 'image_id': int(img_id), 'category': c, 'label': CLS2IDX[c]})
    return pd.DataFrame(rows), reasons, recovered


orig_df, drop_reasons, recovered = audit_labels(
    os.path.join(ORIG_DIR, 'labels.csv'), os.path.join(ORIG_DIR, 'JPEGImages'))

print(f'원본 사용 가능: {len(orig_df)}장  (같은 종류 중복 표기 {recovered}건 복원)')
for k, v in drop_reasons.most_common():
    print(f'  제외 — {k}: {v}건')
display(orig_df['category'].value_counts().reindex(CLASS_NAMES).rename_axis('클래스').reset_index(name='장수'))

In [ ]:
cnt = orig_df['label'].value_counts().reindex(range(NUM_CLASSES)).fillna(0).astype(int).values
print('원본의 클래스 불균형')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {KOR[c]}({c:11s}) {cnt[i]:4d}장 {cnt[i]/cnt.sum()*100:5.1f}%')
print(f'\n불균형 비율 {cnt.max()/max(cnt.min(),1):.1f}배')
print(f'전부 최다 클래스로 찍었을 때 정확도 {cnt.max()/cnt.sum():.1%}')
print('  ↑ 정확도를 주지표로 쓰면 안 되는 이유. 주지표는 macro-F1 으로 한다 (수업 4-5)')

# 증강본 분포
aug_rows = []
for split in ['TRAIN', 'TEST']:
    for c in CLASS_NAMES:
        d = os.path.join(AUG_DIR, split, c)
        for f in sorted(os.listdir(d)):
            aug_rows.append({'path': os.path.join(d, f), 'split': split,
                             'category': c, 'label': CLS2IDX[c]})
aug_df = pd.DataFrame(aug_rows)
piv = aug_df.pivot_table(index='category', columns='split', values='path',
                         aggfunc='count').reindex(CLASS_NAMES)
piv['원본'] = [int(cnt[CLS2IDX[c]]) for c in CLASS_NAMES]
piv['증강배율'] = ((piv['TRAIN'] + piv['TEST']) / piv['원본']).round(1)
display(piv[['원본', 'TRAIN', 'TEST', '증강배율']])
print('증강본 합계', len(aug_df), '장')

### 이 표가 이 프로젝트의 출발점이다

증강본은 **네 클래스가 25%씩으로 균형이 맞아 보인다.**
그런데 **단핵구는 20장을 155배로 불린 것**이고 호중구는 210장을 14배로 불린 것이다.

> **균형이 맞는 것처럼 보이지만 정보량은 여전히 20 대 210이다.**

같은 원본에서 나온 사진 155장은 서로 **다른 사진이 아니라 같은 사진의 다른 각도**다.
이 사실이 §3 의 모든 논의를 만든다.

---
# §3. 누수 진단 — 이 프로젝트의 핵심 ⭐

세 노트북이 각자 다르게 다룬 문제이고, **여기가 점수를 정직하게 만드는 부분**이다.

## 3-0. 무엇이 문제인가

증강본 12,444장은 원본 366장을 불린 것이다. 그래서 이런 일이 벌어질 수 있다.

```
원본 A ──┬─ 증강본 A1  → 학습 데이터
         ├─ 증강본 A2  → 학습 데이터
         └─ 증강본 A3  → 시험 데이터   ← A1, A2 를 외운 모델에게 A3 는 '처음 보는 사진'이 아니다
```

이러면 시험 점수는 **일반화 성능이 아니라 암기력**을 잰 것이 된다.
사용자 요구사항인 *"정답을 외워서 결과를 내놓지 않게"* 가 정확히 이 문제다.

## 3-1. 먼저 시도: 증강본의 부모를 찾을 수 있나?

찾을 수 있다면 **부모 단위로 분할**해서 누수를 원천 차단할 수 있다.
`pro4_1` 노트북이 지각 해시(perceptual hash)로 이걸 시도했다.

**그런데 검증 없이 믿으면 안 된다.** 매칭이 맞는지 확인하는 방법이 있다.

> 증강본의 클래스(폴더 이름)와, 매칭된 부모의 라벨(`labels.csv`)이 **일치해야 한다.**
> 매칭이 무작위라면 일치율은 **부모 라벨 분포에서 기대되는 값(약 25%)** 근처에 머문다.

이 검증을 직접 해 본다.

In [ ]:
# ---- 지각 해시(pHash) 를 numpy 만으로 구현 ------------------------------
# 원리: 이미지를 32x32 흑백으로 줄이고 DCT(주파수 변환) 를 걸어 저주파 8x8 만 남긴 뒤
#       중앙값보다 큰가 아닌가로 63비트 지문을 만든다. 밝기·크기 변화에 강하다.
_N = 32
_n = np.arange(_N)
_C = np.cos(np.pi * (2 * _n[None, :] + 1) * _n[:, None] / (2 * _N))
_C[0] *= np.sqrt(1 / _N); _C[1:] *= np.sqrt(2 / _N)          # DCT-II 행렬

def phash(pil_img):
    g = np.asarray(pil_img.convert('L').resize((_N, _N), Image.BILINEAR), np.float32)
    d = _C @ g @ _C.T
    b = d[:8, :8].flatten()[1:]                              # DC 성분 제외 63비트
    return b > np.median(b)

def color_hist(pil_img, bins=8):
    """검은 회전 패딩을 뺀 RGB 3차원 히스토그램 (회전에 무관)"""
    a = np.asarray(pil_img.convert('RGB').resize((96, 72)), np.float32)
    m = a.max(2) > 25
    q = (a[m] // (256 / bins)).astype(int)
    h = np.zeros(bins ** 3); np.add.at(h, (q[:, 0] * bins + q[:, 1]) * bins + q[:, 2], 1)
    return h / (h.sum() + 1e-9)

# ---- 원본 366장의 지문 (회전 4방향 포함) --------------------------------
orig_paths = sorted(glob.glob(os.path.join(ORIG_DIR, 'JPEGImages', '*.jpg')))
orig_lab_map = dict(zip(orig_df['path'], orig_df['category']))

P_orig, H_orig, owner, olabel = [], [], [], []
for i, p in enumerate(orig_paths):
    im = Image.open(p)
    for r in (0, 90, 180, 270):
        P_orig.append(phash(im.rotate(r, expand=True)))
        owner.append(i)
    H_orig.append(color_hist(im))
    olabel.append(orig_lab_map.get(p))
P_orig = np.array(P_orig, np.uint8); owner = np.array(owner)
H_orig = np.array(H_orig); H_orig /= np.linalg.norm(H_orig, axis=1, keepdims=True) + 1e-9
print('원본 지문', P_orig.shape, '/ 색 히스토그램', H_orig.shape)

In [ ]:
# ---- 증강본 표본 200장을 부모에 매칭해 보고, 라벨 일치율로 신뢰도를 잰다
rs = np.random.RandomState(0)
hit_phash = hit_hist = n_try = 0
for c in CLASS_NAMES:
    d = os.path.join(AUG_DIR, 'TRAIN', c)
    fs = sorted(os.listdir(d))
    for f in rs.choice(fs, 50, replace=False):
        im = Image.open(os.path.join(d, f)); n_try += 1
        q = phash(im).astype(np.uint8)
        j = int(((q[None, :] != P_orig).sum(1)).argmin())         # 해밍 거리 최소
        hit_phash += (olabel[owner[j]] == c)
        h = color_hist(im); h /= np.linalg.norm(h) + 1e-9
        hit_hist += (olabel[int((H_orig @ h).argmax())] == c)

chance = np.mean([ (orig_df['category'] == c).mean() for c in CLASS_NAMES ])
print(f'표본 {n_try}장을 부모에 매칭한 뒤, 부모 라벨과 증강본 클래스가 일치한 비율')
print(f'  지각 해시(pHash)     {hit_phash/n_try*100:5.1f}%')
print(f'  색 히스토그램        {hit_hist/n_try*100:5.1f}%')
print(f'  무작위로 찍었을 기대치 {chance*100:5.1f}%   ← 이 값과 비슷하면 매칭이 무의미하다는 뜻')

### 3-1 결론 — 부모 매칭은 신뢰할 수 없다

일치율이 우연 수준(약 25%)에 머문다. **매칭이 사실상 무작위**라는 뜻이다.

왜 실패하는가:
- 증강본은 원본을 **임의 각도로 회전한 뒤 잘라낸** 것이라 화면에 담긴 적혈구 구성 자체가 다르다
- 지각 해시는 이미지 **전체**의 저주파 구조를 보므로, 잘라낸 조각과 원본은 다른 지문이 된다

> **여기서 중요한 점**: `pro4_1` 은 이 매칭으로 *"TEST 의 77.3%가 TRAIN 과 부모를 공유한다"*고 보고했다.
> 그런데 **매칭이 무작위이면 그 숫자는 자동으로 크게 나온다.**
> TRAIN 9,957장은 366개 부모 거의 전부에 배정되므로, TEST 가 어느 부모에 배정되든 대부분 "공유"로 집계된다.
> 즉 그 77.3%는 누수의 증거가 아니라 **매칭 실패의 부산물**일 가능성이 크다.
>
> 우리는 검증 가능한 지표(라벨 일치율)를 먼저 확인했기 때문에 이 함정을 피할 수 있었다.
> **진단 도구 자체를 먼저 검증한다** — 이것이 이 절의 교훈이다.

## 3-2. 그러면 어떻게 진단하나 — 모델의 행동으로 진단한다

부모를 못 찾으면 분할을 직접 고칠 수는 없다. 대신 **성능 차이로 간접 진단**한다.

같은 모델을 세 가지 시험지로 채점한다.

| 시험지 | 만드는 법 | 누수 위험 |
|---|---|---|
| **① 무작위 검증** | 증강본 TRAIN 을 무작위로 8:2 로 나눈 것 | **높음** — 형제 사진이 양쪽에 들어간다 |
| **② 원본 검증** | 증강되지 않은 원본 354장 | 중간 — 부모일 수는 있으나 증강본은 아니다 |
| **③ 공식 TEST** | `dataset2-master/images/TEST` 폴더 | 낮음(추정) — 제작자가 따로 만든 시험지 |

**진단 규칙**
- ①과 ③이 비슷하다 → 공식 분할도 형제를 공유한다(누수 큼)
- ①이 ③보다 크게 높다 → **①이 부풀려진 것이고, 공식 분할은 대체로 원본 단위로 나뉘어 있다**

> **이전 실행에서 실제로 관측된 값**: 같은 ResNet-18 모델이
> 무작위 검증 **0.9965**, 공식 TEST **0.8536** 을 냈다. 차이가 **0.14**다.
> → 공식 TEST 는 시험지 역할을 하고 있고, **무작위 검증이 부풀려진 쪽**이다.
> (§11 에서 이 노트북의 모델로 다시 측정한다)

## 3-3. 그래서 이 노트북의 검증 정책

```
학습        : 증강본 TRAIN 9,957장
모델 선택   : ② 원본 354장 (증강 없음)   ← 여기가 핵심. ①로 고르면 잘못된 설정을 고른다
조기 종료   : ② 원본 검증의 macro-F1
참고 표시   : ① 무작위 검증도 같이 찍어서 '얼마나 부풀려지는지' 보여 준다
최종 채점   : ③ 공식 TEST — 모든 선택이 끝난 뒤 딱 한 번
```

**왜 ①로 고르면 안 되는가**: 형제 사진이 검증셋에 있으면 "외우기"가 가장 좋은 전략이 된다.
그러면 증강을 끄고 해상도를 낮춘 설정이 1등으로 나온다.
실제로 이전 실행에서 ①을 기준으로 고르자 **"증강 없음(none)이 1등"** 이라는 결론이 나왔다.
이것은 증강이 나쁘다는 뜻이 아니라 **시험지가 잘못됐다는 뜻**이다.

**②를 선택에 쓰는 것이 반칙 아닌가?** 아니다. ②는 최종 채점지(③)가 아니다.
③은 이 노트북 끝까지 한 번도 열지 않는다. 선택에 쓴 자료를 명시하는 것이 정직한 보고다.

---
# §4. 백혈구 ROI 자동 검출 — `pro4_1` 의 핵심 아이디어

## 왜 필요한가

640×480 원본에서 백혈구가 차지하는 면적은 **10% 남짓**이다. 나머지는 적혈구와 배경이고
**모든 클래스에 공통**이다. 화면 전체를 그대로 넣으면 모델이 백혈구가 아니라
**주변 적혈구 배열을 외워도** 점수가 나온다.

여기에 더해 이 노트북에는 **두 번째 이유**가 있다.

> 원본은 640×480 넓은 시야, 증강본은 320×240 좁은 시야다. 화각이 다르다.
> **양쪽 모두 세포 중심으로 잘라내면 화각이 같아진다.**
> 그래야 §3-3 의 "원본으로 모델을 고르고 증강본으로 채점한다"가 성립한다.

## 어떻게 찾나

백혈구 핵은 염색되어 **진한 보라색**이다. 적혈구는 분홍색이다. 그 차이를 쓴다.

- `B - G` (파랑에서 초록을 뺀 값)가 크다 → 보라 계열
- 밝기가 낮다 → 진하다
- 점 잡음을 `binary_opening` 으로 지우고, **가장 큰 연결 성분**을 핵으로 본다
- 핵을 감싸는 정사각형에 여유(margin)를 주면 세포질까지 들어온다

> **📘 수업 3-11 과 이어진다**: 소벨 필터로 경계를 잡던 것과 같은 자리의 고전 영상처리다.
> 학습으로 찾는 게 아니라 **색 규칙으로 찾는다.**

In [ ]:
def detect_wbc_box(rgb, margin=0.35, min_area_frac=0.004, min_half=45):
    """진한 보라색 핵의 최대 연결 성분을 찾아 정사각 ROI 박스를 만든다.
    반환: (x0, y0, x1, y1) 또는 None(검출 실패), 그리고 핵 마스크"""
    a = rgb.astype(np.int16)
    nuc = ((a[..., 2] - a[..., 1]) > 45) & (a[..., 1] < 150)     # 보라 + 어두움
    nuc = ndimage.binary_opening(nuc, np.ones((5, 5)))           # 점 잡음 제거

    lab, n = ndimage.label(nuc)                                   # 연결 성분 라벨링
    if n == 0:
        return None, np.zeros(rgb.shape[:2], bool)
    areas = ndimage.sum(nuc, lab, range(1, n + 1))
    k = int(np.argmax(areas)) + 1                                 # 가장 큰 덩어리 = 핵
    H, W = rgb.shape[:2]
    if areas[k - 1] < min_area_frac * H * W:
        return None, np.zeros(rgb.shape[:2], bool)

    mask = (lab == k)
    ys, xs = np.nonzero(mask)
    cy, cx = (ys.min() + ys.max()) / 2, (xs.min() + xs.max()) / 2
    half = max(ys.max() - ys.min(), xs.max() - xs.min()) * (1 + margin) / 2
    half = max(half, min_half)
    box = (int(max(0, cx - half)), int(max(0, cy - half)),
           int(min(W, cx + half)), int(min(H, cy + half)))
    return box, mask

In [ ]:
def precompute_boxes(paths, cache_path, desc=''):
    """ROI 박스를 한 번만 계산해 CSV 로 캐시한다.
    (매 에폭 검출하면 느리므로 좌표만 미리 구해 두고, 학습할 때는 잘라 쓰기만 한다)"""
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        if len(df) == len(paths):
            print(f'{desc} ROI 캐시 사용: {cache_path} ({len(df)}장)')
            return df
    t0 = time.time(); recs = []
    for i, p in enumerate(paths):
        rgb = np.asarray(Image.open(p).convert('RGB'))
        box, _ = detect_wbc_box(rgb)
        ok = box is not None
        if not ok:
            box = (0, 0, rgb.shape[1], rgb.shape[0])     # 실패하면 전체 이미지를 쓴다
        recs.append({'path': p, 'x0': box[0], 'y0': box[1], 'x1': box[2], 'y1': box[3],
                     'found': int(ok), 'W': rgb.shape[1], 'H': rgb.shape[0]})
        if (i + 1) % 2000 == 0:
            print(f'  {desc} {i+1}/{len(paths)} ... {time.time()-t0:.0f}s', flush=True)
    df = pd.DataFrame(recs); df.to_csv(cache_path, index=False, encoding='utf-8-sig')
    print(f'{desc} ROI 검출 완료 {len(df)}장 / 실패 {(1-df["found"]).sum()}장 / {time.time()-t0:.0f}초')
    return df


box_orig = precompute_boxes(orig_df['path'].tolist(), f'{RESULT_DIR}/roi_orig.csv', '원본')
box_aug  = precompute_boxes(aug_df['path'].tolist(),  f'{RESULT_DIR}/roi_aug.csv',  '증강본')

BOX = {r['path']: (r['x0'], r['y0'], r['x1'], r['y1'])
       for _, r in pd.concat([box_orig, box_aug]).iterrows()}
for name, df_ in [('원본', box_orig), ('증강본', box_aug)]:
    area = ((df_.x1 - df_.x0) * (df_.y1 - df_.y0) / (df_.W * df_.H))
    print(f'{name}: 검출 성공 {df_.found.mean()*100:.1f}% | ROI 면적 비율 중앙값 {area.median()*100:.1f}%')

In [ ]:
# 검출 결과를 눈으로 확인한다 — 원본과 증강본 각각
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for j, c in enumerate(CLASS_NAMES):
    p = orig_df[orig_df.category == c].iloc[0]['path']
    rgb = np.asarray(Image.open(p).convert('RGB')); x0, y0, x1, y1 = BOX[p]
    axes[0, j].imshow(rgb)
    axes[0, j].add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color='lime', lw=2.5))
    axes[0, j].set_title(f'원본 {KOR[c]}'); axes[0, j].axis('off')
    axes[1, j].imshow(rgb[y0:y1, x0:x1]); axes[1, j].set_title('ROI crop'); axes[1, j].axis('off')
plt.suptitle('원본 — 색 기반 백혈구 검출', fontsize=13); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for j, c in enumerate(CLASS_NAMES):
    p = aug_df[(aug_df.category == c) & (aug_df.split == 'TRAIN')].iloc[0]['path']
    rgb = np.asarray(Image.open(p).convert('RGB')); x0, y0, x1, y1 = BOX[p]
    axes[0, j].imshow(rgb)
    axes[0, j].add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color='lime', lw=2.5))
    axes[0, j].set_title(f'증강본 {KOR[c]}'); axes[0, j].axis('off')
    axes[1, j].imshow(rgb[y0:y1, x0:x1]); axes[1, j].set_title('ROI crop'); axes[1, j].axis('off')
plt.suptitle('증강본 — 같은 검출기로 화각을 맞춘다', fontsize=13); plt.tight_layout(); plt.show()

---
# §5. 데이터셋 · 분할 · 증강

## 5-1. ROI 를 미리 잘라 메모리에 올린다

디스크에서 매번 읽고 자르면 에폭마다 시간이 낭비된다.
**한 번만 잘라 `IMAGE_SIZE` 로 줄인 뒤 `uint8` 배열로 메모리에 올린다** (`pro4_1` 방식).

메모리가 부족하면 `PRELOAD = False` 로 바꾸면 매번 디스크에서 읽는다(느리지만 메모리를 안 쓴다).

In [ ]:
PRELOAD = True
USE_ROI_DEFAULT = True

est_mb = (len(aug_df) + len(orig_df)) * IMAGE_SIZE * IMAGE_SIZE * 3 / 1e6
print(f'전부 메모리에 올리면 약 {est_mb:.0f} MB 필요 (uint8 기준)')

def load_crop(path, use_roi=True, size=IMAGE_SIZE):
    """이미지를 열어 ROI 로 자르고 size x size 로 줄여 uint8 배열로 돌려준다"""
    im = Image.open(path).convert('RGB')
    if use_roi:
        x0, y0, x1, y1 = BOX[path]
        im = im.crop((x0, y0, x1, y1))
    return np.asarray(im.resize((size, size), Image.BILINEAR), dtype=np.uint8)


def preload(paths, use_roi, desc=''):
    t0 = time.time()
    arr = np.zeros((len(paths), IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        arr[i] = load_crop(p, use_roi)
    print(f'  {desc} {len(paths)}장 적재 {time.time()-t0:.0f}초 ({arr.nbytes/1e6:.0f}MB)')
    return arr

if PRELOAD:
    print('ROI 적용본 적재')
    X_orig_roi = preload(orig_df['path'].tolist(), True,  '원본(ROI)')
    X_aug_roi  = preload(aug_df['path'].tolist(),  True,  '증강본(ROI)')
    print('전체 이미지본 적재 (ROI 효과를 비교하기 위한 대조군)')
    X_orig_full = preload(orig_df['path'].tolist(), False, '원본(전체)')
    X_aug_full  = preload(aug_df['path'].tolist(),  False, '증강본(전체)')
else:
    X_orig_roi = X_aug_roi = X_orig_full = X_aug_full = None
    print('PRELOAD=False — 매번 디스크에서 읽는다')

## 5-2. 증강 — 무엇을 주고 무엇을 안 주는가

> **📘 수업 3-13 / 5-2**: 증강은 **학습용에만** 건다. **검증·시험 데이터는 절대 증강하지 않는다.**
> (수업 5장에서 검증셋에 증강이 걸려 있던 버그를 지적했었다. 여기서는 고쳐 쓴다)

| 증강 | 판단 | 이유 |
|---|---|---|
| 90도 회전 · 상하/좌우 뒤집기 | **쓴다** | 혈액 도말은 **방향에 의미가 없다**. 90도 단위라 **검은 패딩이 안 생긴다** (`pro4_1` 방식) |
| 임의 각도 회전 · 이동 · 확대 | **실험으로 판정** | 증강본에는 이미 걸려 있다. 원본에는 안 걸려 있다 |
| 색조 변화(ColorJitter) | **실험으로 판정** | 염색 색조가 클래스 신호(호산구 과립)다. 다만 **검사실마다 염색이 다르므로** 약하게 주면 일반화에 도움이 될 수 있다 |
| RandomErasing | **안 쓴다** | 화면에 판단 근거가 세포 하나뿐인데 가리면 라벨과 무관한 이미지가 된다 |

`pro4_1` 은 색 증강을 아예 배제했고, 이전 노트북은 약하게 줬다.
**어느 쪽이 맞는지는 §11 에서 원본 검증셋으로 판정한다.**

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class RandomRot90:
    """0/90/180/270 도 중 하나로 회전. 각도가 90도 배수라 검은 패딩이 생기지 않는다"""
    def __call__(self, img):
        k = random.randint(0, 3)
        return img if k == 0 else img.rotate(90 * k, expand=True)

AUG_PRESETS = {
    'none'      : dict(rot90=False, flip=False, affine=0,  color=0.0),
    'flip'      : dict(rot90=True,  flip=True,  affine=0,  color=0.0),   # pro4_1 방식
    'flip_geo'  : dict(rot90=True,  flip=True,  affine=12, color=0.0),
    'flip_color': dict(rot90=True,  flip=True,  affine=0,  color=0.15),
    'full'      : dict(rot90=True,  flip=True,  affine=12, color=0.15),
    'strong'    : dict(rot90=True,  flip=True,  affine=20, color=0.30),
}

def build_train_tf(preset):
    p = AUG_PRESETS[preset]; ops = []
    if p['rot90']: ops.append(RandomRot90())
    if p['flip']:  ops += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip()]
    if p['affine'] > 0:
        ops.append(transforms.RandomAffine(degrees=p['affine'], translate=(0.05, 0.05),
                                           scale=(0.9, 1.1)))
    if p['color'] > 0:
        c = p['color']
        ops.append(transforms.ColorJitter(brightness=c, contrast=c, saturation=c*0.7, hue=c*0.15))
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

EVAL_TF = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def denorm(x):
    m = torch.tensor(IMAGENET_MEAN).view(3,1,1); s = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (x.detach().cpu() * s + m).clamp(0,1).permute(1,2,0).numpy()

# 증강 프리셋을 눈으로 확인
src = Image.fromarray(X_aug_roi[0]) if PRELOAD else Image.fromarray(load_crop(aug_df.iloc[0]['path']))
fig, axes = plt.subplots(len(AUG_PRESETS), 5, figsize=(11, 2.1*len(AUG_PRESETS)))
for r, name in enumerate(AUG_PRESETS):
    tf_ = build_train_tf(name); random.seed(0); torch.manual_seed(0)
    for k in range(5):
        axes[r, k].imshow(denorm(tf_(src))); axes[r, k].axis('off')
    axes[r, 0].set_title(name, loc='left', fontsize=11)
plt.suptitle('증강 프리셋 (ROI crop 위에 적용)'); plt.tight_layout(); plt.show()

## 5-3. Dataset 클래스

> **📘 수업 2-4 / 4-3**: `Dataset` 은 `__len__` 과 `__getitem__` 두 개만 있으면 된다.
> 라벨은 **`long`(정수)** 으로 준다 — `CrossEntropyLoss` 가 정수 라벨을 받기 때문이다.
> (4·5장의 이진 분류에서는 `float` 로 바꿔야 했다. 여기서는 그럴 필요가 없다)

In [ ]:
class WBCDataset(Dataset):
    def __init__(self, images, labels, transform, paths=None, use_roi=True):
        self.images = images          # 미리 적재한 uint8 배열 (PRELOAD=True) 또는 None
        self.paths = paths            # PRELOAD=False 일 때 사용
        self.labels = np.asarray(labels, dtype=np.int64)
        self.transform = transform
        self.use_roi = use_roi

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        if self.images is not None:
            arr = self.images[i]
        else:
            arr = load_crop(self.paths[i], self.use_roi)
        x = self.transform(Image.fromarray(arr))
        return x, torch.tensor(self.labels[i], dtype=torch.long)


def make_loader(ds, batch_size=None, shuffle=False):
    return DataLoader(ds, batch_size=batch_size or BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'),
                      drop_last=shuffle and len(ds) > BATCH_SIZE,
                      persistent_workers=(NUM_WORKERS > 0))

## 5-4. 세 개의 시험지를 만든다

> **📘 수업 4-3**: `train_test_split(stratify=y)` — **층화 분할**. 클래스 비율을 양쪽에서 유지한다.

§3-3 에서 정한 정책대로 만든다.

- **학습** : 증강본 TRAIN 의 80%
- **① 무작위 검증** : 증강본 TRAIN 의 20% — *부풀려짐을 보여 주는 용도*
- **② 원본 검증** : 원본 354장 전체 — **모델 선택과 조기 종료의 기준**
- **③ 공식 TEST** : 증강본 TEST 폴더 — §16 까지 열지 않는다

In [ ]:
aug_train_mask = (aug_df['split'] == 'TRAIN').values
aug_test_mask   = (aug_df['split'] == 'TEST').values
y_aug = aug_df['label'].values
idx_trainpool = np.where(aug_train_mask)[0]
idx_test      = np.where(aug_test_mask)[0]

idx_fit, idx_randval = train_test_split(
    idx_trainpool, test_size=0.2, stratify=y_aug[idx_trainpool], random_state=42)

y_orig = orig_df['label'].values

print(f'학습(fit)        {len(idx_fit):5d}장  (증강본 TRAIN 의 80%)')
print(f'① 무작위 검증     {len(idx_randval):5d}장  (증강본 TRAIN 의 20%) — 진단용')
print(f'② 원본 검증       {len(y_orig):5d}장  (증강되지 않은 원본)     — 모델 선택 기준')
print(f'③ 공식 TEST      {len(idx_test):5d}장  (증강본 TEST 폴더)      — 마지막에 딱 한 번')
print()
print('클래스 분포')
display(pd.DataFrame({
    '학습':        np.bincount(y_aug[idx_fit],     minlength=4),
    '①무작위검증': np.bincount(y_aug[idx_randval], minlength=4),
    '②원본검증':   np.bincount(y_orig,             minlength=4),
    '③공식TEST':   np.bincount(y_aug[idx_test],    minlength=4),
}, index=[f'{c}({KOR[c]})' for c in CLASS_NAMES]))

In [ ]:
def build_datasets(preset='flip', use_roi=True, subset=None, seed=42):
    """학습/①/②/③ 네 개의 Dataset 을 한꺼번에 만든다"""
    Xa = (X_aug_roi if use_roi else X_aug_full) if PRELOAD else None
    Xo = (X_orig_roi if use_roi else X_orig_full) if PRELOAD else None
    train_tf = build_train_tf(preset)
    ap = aug_df['path'].values; op = orig_df['path'].values

    fit_idx = idx_fit
    if subset:
        rs = np.random.RandomState(seed)
        fit_idx = rs.choice(idx_fit, min(subset, len(idx_fit)), replace=False)

    ds_fit  = WBCDataset(Xa[fit_idx] if PRELOAD else None, y_aug[fit_idx], train_tf,
                         ap[fit_idx], use_roi)
    ds_rand = WBCDataset(Xa[idx_randval] if PRELOAD else None, y_aug[idx_randval], EVAL_TF,
                         ap[idx_randval], use_roi)
    ds_orig = WBCDataset(Xo if PRELOAD else None, y_orig, EVAL_TF, op, use_roi)
    ds_test = WBCDataset(Xa[idx_test] if PRELOAD else None, y_aug[idx_test], EVAL_TF,
                         ap[idx_test], use_roi)
    return ds_fit, ds_rand, ds_orig, ds_test


ds_fit, ds_rand, ds_orig, ds_test = build_datasets('flip', True)
xb, yb = next(iter(make_loader(ds_fit, shuffle=True)))
print('배치 모양', tuple(xb.shape), '| 라벨 dtype', yb.dtype, '| 예시', yb[:8].tolist())

---
# §6. 모델 6종

> **📘 수업 2-3**: `nn.Module` 을 상속하고 `__init__` 에 층을, `forward` 에 흐름을 쓴다.
> **출력층에 softmax 를 붙이지 않는다** — `CrossEntropyLoss` 가 안에서 처리한다.

| 모델 | 수업 | 무엇을 재는가 |
|---|---|---|
| **M0** 수작업 특징 + 로지스틱회귀 | 영상처리 | 딥러닝의 **추가 가치** (§9) |
| **M1** SimpleCNN | 3-12 | 밑바닥 CNN |
| **M2** CNN + BatchNorm + Dropout | 2-10, 3-12 | 정규화 효과 |
| **M3** MyResNet18 (잔차 연결 직접 구현) | 3-14 | 구조를 깊게 하면? |
| **M4** 전이학습 — 백본 고정 | 4-6 | ImageNet 특징만으로? |
| **M5** 전이학습 — 전체 미세조정 | 5-3 | 본 게임 |

**M3 와 M5 는 구조가 같고 사전학습 유무만 다르다.** 이 둘의 차이가 전이학습의 순수 이득이다.
여섯 모델 모두 **`특징맵 → GAP → Linear`** 로 끝난다. 이 구조여야 §18 의 CAM 을 뽑을 수 있다.

In [ ]:
class SimpleCNN(nn.Module):
    """M1 — 수업 3-12 의 기본 CNN (Conv-ReLU-Pool 을 4단)"""
    def __init__(self, num_classes=NUM_CLASSES, width=32, use_bn=False, dropout=0.0):
        super().__init__()
        def block(i, o):
            layers = [nn.Conv2d(i, o, 3, padding=1)]
            if use_bn: layers.append(nn.BatchNorm2d(o))
            # ★ inplace=False : Grad-CAM 의 backward 후크가 inplace 연산과 충돌한다 (pro4_1 의 지적)
            layers += [nn.ReLU(inplace=False), nn.MaxPool2d(2)]
            if dropout > 0: layers.append(nn.Dropout2d(dropout))
            return layers
        self.features = nn.Sequential(*block(3, width), *block(width, width*2),
                                      *block(width*2, width*4), *block(width*4, width*8))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(width*8, num_classes)

    def forward_features(self, x): return self.features(x)
    def forward(self, x):
        return self.head(self.drop(self.pool(self.forward_features(x)).flatten(1)))


class BasicBlock(nn.Module):
    """수업 3-14 의 잔차 블록.  y = F(x) + x
    모양이 달라질 때만 1x1 합성곱으로 맞춘다 — shortcut 이 조건부인 이유"""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                          nn.BatchNorm2d(out_ch))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=False)
        out = self.bn2(self.conv2(out)) + self.shortcut(x)      # ← 잔차 연결
        return F.relu(out, inplace=False)


class MyResNet18(nn.Module):
    """M3 — 수업 3-14 에서 조립한 ResNet18 (사전학습 없음) + 3-15 의 He 초기화"""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
                                  nn.BatchNorm2d(64), nn.ReLU(inplace=False),
                                  nn.MaxPool2d(3, stride=2, padding=1))
        self.layer1 = nn.Sequential(BasicBlock(64, 64),      BasicBlock(64, 64))
        self.layer2 = nn.Sequential(BasicBlock(64, 128, 2),  BasicBlock(128, 128))
        self.layer3 = nn.Sequential(BasicBlock(128, 256, 2), BasicBlock(256, 256))
        self.layer4 = nn.Sequential(BasicBlock(256, 512, 2), BasicBlock(512, 512))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(512, num_classes)
        for m in self.modules():                       # 수업 3-15: He(kaiming) 초기화
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward_features(self, x):
        return self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
    def forward(self, x):
        return self.head(self.pool(self.forward_features(x)).flatten(1))

In [ ]:
class TransferNet(nn.Module):
    """M4(freeze=True) / M5(freeze=False) — 수업 4-6, 5-3.
    torchvision 사전학습 백본에서 분류층을 떼고 4클래스 선형층을 붙인다"""
    BUILDERS = {
        'resnet18':          (models.resnet18,          'ResNet18_Weights'),
        'resnet34':          (models.resnet34,          'ResNet34_Weights'),
        'efficientnet_b0':   (models.efficientnet_b0,   'EfficientNet_B0_Weights'),
        'efficientnet_v2_s': (models.efficientnet_v2_s, 'EfficientNet_V2_S_Weights'),
        'mobilenet_v3_small':(models.mobilenet_v3_small,'MobileNet_V3_Small_Weights'),
    }
    def __init__(self, backbone='efficientnet_v2_s', num_classes=NUM_CLASSES,
                 pretrained=True, freeze=False, dropout=0.3, probe_size=224):
        super().__init__()
        fn, wname = self.BUILDERS[backbone]
        weights = getattr(models, wname).IMAGENET1K_V1 if pretrained else None
        net = fn(weights=weights)
        if backbone.startswith('resnet'):
            self.features = nn.Sequential(*list(net.children())[:-2])   # avgpool·fc 제거
        else:
            self.features = net.features                                # efficientnet·mobilenet
        if freeze:                                    # 수업 4-6: 백본 고정 -> 헤드만 학습
            for p in self.features.parameters():
                p.requires_grad = False
        with torch.no_grad():                         # 출력 채널 수는 더미 입력으로 확인 (손으로 적지 않는다)
            n_feat = self.features(torch.zeros(1, 3, probe_size, probe_size)).shape[1]
        self.num_features = n_feat
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(n_feat, num_classes)

    def forward_features(self, x): return self.features(x)
    def forward(self, x):
        return self.head(self.drop(self.pool(self.forward_features(x)).flatten(1)))


def build_model(name, pretrained=True, freeze=False, dropout=0.0, image_size=IMAGE_SIZE):
    if name == 'simplecnn':   return SimpleCNN(use_bn=False, dropout=0.0)
    if name == 'cnn_bn_drop': return SimpleCNN(use_bn=True,  dropout=max(dropout, 0.25))
    if name == 'myresnet18':  return MyResNet18()
    return TransferNet(name, pretrained=pretrained, freeze=freeze,
                       dropout=dropout, probe_size=image_size)

def count_params(m):
    tot = sum(p.numel() for p in m.parameters())
    trn = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tot, trn

In [ ]:
# 수업 5-3 의 습관: 만들자마자 더미 입력을 흘려 shape 을 확인한다
print(f"{'모델':30s}{'특징맵':>18s}{'출력':>9s}{'전체 파라미터':>16s}{'학습 파라미터':>16s}")
for label, kw in [('M1 SimpleCNN',            dict(name='simplecnn',   pretrained=False)),
                  ('M2 CNN+BN+Dropout',       dict(name='cnn_bn_drop', pretrained=False)),
                  ('M3 MyResNet18(잔차)',      dict(name='myresnet18',  pretrained=False)),
                  ('M4 EffNetV2-S 고정',       dict(name='efficientnet_v2_s', freeze=True)),
                  ('M5 EffNetV2-S 미세조정',    dict(name='efficientnet_v2_s', freeze=False))]:
    nm = kw.pop('name')
    m = build_model(nm, image_size=IMAGE_SIZE, **kw)
    with torch.no_grad():
        f = m.forward_features(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE))
        o = m(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE))
    tot, trn = count_params(m)
    print(f'{label:30s}{str(tuple(f.shape)):>18s}{str(tuple(o.shape)):>9s}{tot:>16,}{trn:>16,}')
    del m

**수업 4-6 에서 가장 인상적이었던 숫자가 여기서도 나온다.**
백본을 고정하면 학습 파라미터가 **수천 개**로 줄어든다(전체의 0.03% 수준).
그만큼 빠르지만, ImageNet 특징이 현미경 이미지에 맞지 않으면 성능이 따라오지 못한다. §12 에서 확인한다.

---
# §7. 학습 · 평가 함수

> **📘 수업 2-5 — 학습 루프 다섯 줄. 이 순서가 전부다.**
> ```python
> optimizer.zero_grad()               # 1. 이전 기울기 지우기
> output = model(data)                # 2. 순전파
> loss = criterion(output, target)    # 3. 손실 계산
> loss.backward()                     # 4. 역전파
> optimizer.step()                    # 5. 가중치 갱신
> ```
> **`zero_grad()` 를 빼면 기울기가 누적된다.** 파이토치에서 가장 많이 하는 실수다.
>
> **📘 수업 2-8 — 평가할 때 반드시 두 가지**: `model.eval()` + `torch.no_grad()`.
> 하는 일이 다르므로 둘 다 써야 한다.
>
> **📘 수업 2-1 정정**: 수업 `2.ipynb` 는 에폭마다 **마지막 배치의 손실**만 출력해 값이 들쭉날쭉했다.
> 여기서는 **에폭 평균**을 낸다.

### 이진 분류(4·5장) → 4클래스 다중 분류 대응표

| | 4·5장 (이진) | 이 프로젝트 (4클래스) |
|---|---|---|
| 출력층 | `nn.Linear(n, 1)` | **`nn.Linear(n, 4)`** |
| 손실 | `BCEWithLogitsLoss(pos_weight=)` | **`CrossEntropyLoss(weight=)`** |
| 로짓 | `model(x).squeeze(1)` | **`model(x)`** — squeeze 안 함 |
| 확률 | `torch.sigmoid(logits)` | **`torch.softmax(logits, dim=1)`** |
| 예측 | `probs > 0.5` | **`probs.argmax(1)`** |
| 라벨 | `.float()` 필요 | **`long` 그대로** |
| 주지표 | AUC | **macro-F1** (클래스 4개 + 원본이 불균형) |

In [ ]:
@torch.no_grad()                                     # ← 수업 2-8
def evaluate(model, loader, criterion=None):
    model.eval()                                     # ← 수업 2-8
    losses, L, T = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits = model(xb)                           # (B, 4)
        if criterion is not None:
            losses.append(criterion(logits, yb).item() * len(yb))
        L.append(logits.float().cpu().numpy()); T.append(yb.cpu().numpy())

    logits = np.concatenate(L); trues = np.concatenate(T)
    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    preds = probs.argmax(1)
    try:
        auc = roc_auc_score(trues, probs, multi_class='ovr', average='macro')
    except ValueError:
        auc = float('nan')
    return {'loss': (sum(losses)/len(trues)) if criterion is not None else float('nan'),
            'accuracy': accuracy_score(trues, preds),
            'macro_f1': f1_score(trues, preds, average='macro', zero_division=0),
            'balanced': balanced_accuracy_score(trues, preds),
            'macro_prec': precision_score(trues, preds, average='macro', zero_division=0),
            'macro_rec': recall_score(trues, preds, average='macro', zero_division=0),
            'recall_per_class': recall_score(trues, preds, average=None,
                                             labels=range(NUM_CLASSES), zero_division=0),
            'auc': auc, 'logits': logits, 'probs': probs, 'trues': trues, 'preds': preds}


def class_weights_from(labels):
    """수업 4-7 의 pos_weight 를 다중 클래스로 옮긴 것 — 적은 클래스에 큰 가중치"""
    cnt = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float32)
    return len(labels) / (NUM_CLASSES * np.maximum(cnt, 1))

In [ ]:
def train_model(model, fit_loader, monitor_loader, epochs=10, lr=1e-3, weight_decay=0.0,
                class_weight=None, scheduler='plateau', patience=5, amp=True,
                ckpt_path='./results/ckpt/tmp.pt', label='model', verbose=True,
                extra_loaders=None):
    """수업 2-5 의 다섯 줄에 스케줄러·체크포인트·조기종료·시간 측정을 붙인 것.

    monitor_loader : 조기종료와 체크포인트의 기준이 되는 검증셋 (우리는 '② 원본 검증')
    extra_loaders  : 같이 찍어만 볼 검증셋들 {'이름': loader}  (우리는 '① 무작위 검증')
    """
    model = model.to(device)
    w = None if class_weight is None else torch.tensor(class_weight, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=w)
    params = [p for p in model.parameters() if p.requires_grad]     # 고정 백본이면 헤드만
    optimizer = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)   # 수업 2·4장 = Adam

    # 수업 2-9 / 3-15 : 스케줄러 세 가지
    if scheduler == 'cosine':
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    elif scheduler == 'step':
        sched = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs//3), gamma=0.3)
    else:
        # macro-F1 을 기준으로 하므로 mode='max'. (수업 2.ipynb 는 학습 손실을 넣는 실수가 있었다)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.3,
                                                           patience=2, min_lr=1e-7)
    use_amp = bool(amp and device.type == 'cuda')     # 혼합정밀 — 3분 예산을 맞추는 가장 싼 수단
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    hist = collections.defaultdict(list)
    best, best_epoch, bad = -np.inf, 0, 0

    for epoch in range(1, epochs + 1):
        model.train()                                # ← 수업 2-8: 학습 모드
        run_loss, run_correct, n = 0.0, 0, 0
        if device.type == 'cuda': torch.cuda.synchronize()      # ← 수업 1장
        t0 = time.time()

        for xb, yb in fit_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            # ===== 수업 2-5. 학습 루프 다섯 줄 =====
            optimizer.zero_grad(set_to_none=True)                        # 1
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(xb)                                       # 2
                loss = criterion(logits, yb)                             # 3
            scaler.scale(loss).backward()                                # 4
            scaler.step(optimizer); scaler.update()                      # 5
            # =====================================
            run_loss += loss.item() * len(yb)
            run_correct += (logits.argmax(1) == yb).sum().item()
            n += len(yb)

        m = evaluate(model, monitor_loader, criterion)
        if device.type == 'cuda': torch.cuda.synchronize()
        sec = time.time() - t0

        sched.step(m['macro_f1']) if scheduler == 'plateau' else sched.step()

        hist['loss'].append(run_loss/n); hist['accuracy'].append(run_correct/n)
        hist['val_loss'].append(m['loss']); hist['val_acc'].append(m['accuracy'])
        hist['val_macro_f1'].append(m['macro_f1'])
        hist['lr'].append(optimizer.param_groups[0]['lr']); hist['sec'].append(sec)

        extra_txt = ''
        if extra_loaders:
            for nm, ld in extra_loaders.items():
                me = evaluate(model, ld)
                hist[f'{nm}_macro_f1'].append(me['macro_f1'])
                extra_txt += f' | {nm} f1 {me["macro_f1"]:.4f}'

        star = ''
        if m['macro_f1'] > best:
            best, best_epoch, bad = m['macro_f1'], epoch, 0
            torch.save(model.state_dict(), ckpt_path)     # ← 수업 2-6: state_dict 만 저장
            star = '저장'
        else:
            bad += 1

        if verbose:
            over = ' ⚠3분초과' if sec > EPOCH_BUDGET else ''
            print(f'[{label}] {epoch:2d}/{epochs} loss {run_loss/n:.4f} acc {run_correct/n:.4f} '
                  f'| 검증 f1 {m["macro_f1"]:.4f} acc {m["accuracy"]:.4f}{extra_txt} '
                  f'| lr {optimizer.param_groups[0]["lr"]:.1e} ({sec:.0f}s){over} {star}')

        if bad >= patience:
            if verbose: print(f'  조기종료: {patience} 에폭 동안 개선 없음')
            break

    hist = dict(hist)
    hist.update(best_epoch=best_epoch, best_score=best, ckpt_path=ckpt_path,
                epoch_sec_mean=float(np.mean(hist['sec'])), epochs_run=len(hist['loss']))
    if verbose:
        print(f'best 검증 macro-F1 = {best:.4f} @epoch {best_epoch} → {ckpt_path}')
        print(f'에폭당 평균 {hist["epoch_sec_mean"]:.0f}초 (예산 {EPOCH_BUDGET}초) '
              f'| 총 {hist["epochs_run"]}에폭 {sum(hist["sec"])/60:.1f}분')
    return hist

### 실험 기록 — "무한 반복"을 감당하는 장치

실험을 수십 번 돌리므로 결과를 사람이 기억하지 않게 만든다.

- 모든 실험이 `results/runs.csv` 에 **한 줄씩 누적**
- **이미 돌린 `run_id` 는 자동으로 건너뛴다** → 노트북을 다시 실행해도 이어서 진행
- 예측 확률을 `results/preds/*.npz` 에 저장 → **§21 의 McNemar 검정은 재학습 없이** 가능

In [ ]:
RUNS_CSV = f'{RESULT_DIR}/runs.csv'
if os.path.exists(RUNS_CSV):
    print(f'⚠ runs.csv 에 이미 {len(pd.read_csv(RUNS_CSV, encoding="utf-8-sig"))}건이 있다. '
          '다른 버전으로 돌린 기록이면 results/ 를 지우고 시작하는 것이 안전하다.')

def already_run(run_id):
    if not os.path.exists(RUNS_CSV): return False
    with open(RUNS_CSV, newline='', encoding='utf-8-sig') as f:
        return any(r.get('run_id') == run_id for r in csv.DictReader(f))

def log_run(run_id, params, metrics_by_tag, hist=None):
    row = dict(run_id=run_id, **params)
    for tag, m in metrics_by_tag.items():
        if m is None: continue
        for k in ['accuracy', 'macro_f1', 'balanced', 'macro_prec', 'macro_rec', 'auc', 'loss']:
            row[f'{tag}_{k}'] = round(float(m[k]), 5)
    if hist:
        row.update(best_epoch=hist['best_epoch'], epochs_run=hist['epochs_run'],
                   epoch_sec=round(hist['epoch_sec_mean'], 1))
    save = {}
    for tag, m in metrics_by_tag.items():
        if m is not None:
            save[f'{tag}_probs'] = m['probs']; save[f'{tag}_trues'] = m['trues']
    if save: np.savez_compressed(f'{RESULT_DIR}/preds/{run_id}.npz', **save)

    old = []
    if os.path.exists(RUNS_CSV):
        with open(RUNS_CSV, newline='', encoding='utf-8-sig') as f: old = list(csv.DictReader(f))
    keys = sorted(set().union(*[set(r) for r in old + [row]])) if old else list(row)
    with open(RUNS_CSV, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
        for r in old + [row]: w.writerow(r)
    return row

def runs_table():
    return pd.read_csv(RUNS_CSV, encoding='utf-8-sig') if os.path.exists(RUNS_CSV) else pd.DataFrame()

def summarize(m, name=''):
    return (f"{name:22s} acc {m['accuracy']:.4f}  macroF1 {m['macro_f1']:.4f}  "
            f"balanced {m['balanced']:.4f}  AUC {m['auc']:.4f}")

In [ ]:
def run_experiment(run_id, model_name='efficientnet_v2_s', preset='flip', use_roi=True,
                   lr=1e-4, epochs=None, freeze=False, pretrained=True, dropout=0.3,
                   weight_decay=0.0, scheduler='plateau', use_class_weight=False,
                   seed=42, subset=None, patience=4, batch_size=None, force=False, verbose=True):
    """실험 한 건.
    조기종료·체크포인트 기준 = ② 원본 검증 macro-F1   (§3-3 의 정책)
    ① 무작위 검증은 '얼마나 부풀려지는지' 보려고 같이 찍기만 한다.
    ③ 공식 TEST 는 여기서 절대 건드리지 않는다."""
    if already_run(run_id) and not force:
        if verbose: print(f'[건너뜀] {run_id} — 이미 runs.csv 에 있음')
        return None
    epochs = epochs or SCREEN_EPOCHS
    set_seed(seed)

    d_fit, d_rand, d_orig, _ = build_datasets(preset, use_roi, subset=subset, seed=seed)
    L_fit  = make_loader(d_fit, batch_size, shuffle=True)
    L_rand = make_loader(d_rand, (batch_size or BATCH_SIZE) * 2)
    L_orig = make_loader(d_orig, (batch_size or BATCH_SIZE) * 2)

    model = build_model(model_name, pretrained=pretrained, freeze=freeze,
                        dropout=dropout, image_size=IMAGE_SIZE)
    tot, trn = count_params(model)
    cw = class_weights_from(d_fit.labels) if use_class_weight else None
    if verbose:
        print(f'=== {run_id} | {model_name} | 증강 {preset} | ROI {use_roi} | lr {lr:g} '
              f'| seed {seed} | 학습 파라미터 {trn:,}/{tot:,}')

    hist = train_model(model, L_fit, L_orig, epochs=epochs, lr=lr, weight_decay=weight_decay,
                       class_weight=cw, scheduler=scheduler, patience=patience,
                       ckpt_path=f'{RESULT_DIR}/ckpt/{run_id}.pt', label=run_id,
                       verbose=verbose, extra_loaders={'①무작위': L_rand})
    model.load_state_dict(torch.load(hist['ckpt_path'], map_location=device))   # 최적 시점 복원
    m_orig = evaluate(model, L_orig, nn.CrossEntropyLoss())
    m_rand = evaluate(model, L_rand, nn.CrossEntropyLoss())

    log_run(run_id,
            dict(model=model_name, preset=preset, use_roi=int(use_roi), lr=lr, freeze=int(freeze),
                 pretrained=int(pretrained), dropout=dropout, weight_decay=weight_decay,
                 scheduler=scheduler, class_weight=int(use_class_weight), seed=seed,
                 subset=subset or 0, batch_size=batch_size or BATCH_SIZE,
                 params_M=round(tot/1e6, 2)),
            {'orig': m_orig, 'rand': m_rand}, hist)
    if verbose:
        print(summarize(m_orig, '  ② 원본검증'))
        print(summarize(m_rand, '  ① 무작위검증') + '   ← 부풀려진 쪽')
    return dict(model=model, history=hist, orig=m_orig, rand=m_rand)


def plot_history(hist, title=''):
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.3))
    e = range(1, len(hist['loss']) + 1)
    ax[0].plot(e, hist['loss'], marker='o', label='학습'); ax[0].plot(e, hist['val_loss'], marker='o', label='②원본검증')
    ax[0].set_title(f'{title} 손실'); ax[0].legend()
    ax[1].plot(e, hist['accuracy'], marker='o', label='학습'); ax[1].plot(e, hist['val_acc'], marker='o', label='②원본검증')
    ax[1].set_title('정확도'); ax[1].legend()
    ax[2].plot(e, hist['val_macro_f1'], marker='o', color='tab:green', label='②원본검증')
    if '①무작위_macro_f1' in hist:
        ax[2].plot(e, hist['①무작위_macro_f1'], marker='s', color='tab:red', label='①무작위검증')
    ax[2].set_title('macro-F1 — 두 시험지 비교'); ax[2].legend()
    ax[3].plot(e, hist['lr'], marker='o', color='tab:purple'); ax[3].set_yscale('log'); ax[3].set_title('학습률')
    for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def plot_confusion(y_true, y_pred, normalize=False, title='혼동행렬'):
    """수업 2-8 / 5-4. 5장에서 지적된 '정규화를 imshow 앞으로' 를 반영"""
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    shown = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1) if normalize else cm
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    im_ = ax.imshow(shown, cmap='Blues'); fig.colorbar(im_, ax=ax); ax.set_title(title)
    t = np.arange(NUM_CLASSES)
    ax.set_xticks(t); ax.set_xticklabels([KOR[c] for c in CLASS_NAMES], rotation=25, ha='right')
    ax.set_yticks(t); ax.set_yticklabels([KOR[c] for c in CLASS_NAMES])
    thr = shown.max()/2
    for i, j in itertools.product(range(NUM_CLASSES), range(NUM_CLASSES)):
        ax.text(j, i, f'{shown[i,j]:.2f}' if normalize else f'{cm[i,j]}',
                ha='center', color='white' if shown[i,j] > thr else 'black')
    ax.set_ylabel('정답'); ax.set_xlabel('예측'); plt.tight_layout(); plt.show()
    return cm

---
# §8. 속도 프로브 — "1 에폭 최대 3분"

**모델을 고르기 전에 시간을 먼저 잰다.** 고르고 나서 재면 조건을 못 맞췄을 때 처음부터 다시 해야 한다.

> **📘 수업 1장**: GPU 연산은 **비동기**다. `torch.cuda.synchronize()` 를 불러 실제로 끝나기를 기다려야
> 시간이 제대로 측정된다.

> 수업 5장에서 ResNetV2-50 은 4,173장에 **200초/에폭**이었다. 우리 학습셋은 약 7,966장으로 두 배다.
> 같은 속도라면 380초 → **조건 위반**. "수업에서 쓴 모델"을 그대로 가져오면 안 된다는 뜻이다.

In [ ]:
N_FIT = len(idx_fit)

def speed_probe(model_name, batch_size=BATCH_SIZE, n_batches=8, freeze=False, n_train=None):
    set_seed(0)
    n_train = n_train or N_FIT
    model = build_model(model_name, pretrained=False, freeze=freeze, image_size=IMAGE_SIZE).to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    crit = nn.CrossEntropyLoss()
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    x = torch.randn(batch_size, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    y = torch.randint(0, NUM_CLASSES, (batch_size,), device=device)

    model.train()
    for i in range(n_batches + 3):
        if i == 3:
            if device.type == 'cuda': torch.cuda.synchronize()
            t0 = time.time()
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    if device.type == 'cuda': torch.cuda.synchronize()
    dt = time.time() - t0

    ips = n_batches * batch_size / dt
    est = n_train / ips * 1.25                      # 검증·데이터로딩 몫 25% 가산
    tot, _ = count_params(model)
    del model, opt
    if device.type == 'cuda': torch.cuda.empty_cache()
    return dict(model=model_name, params_M=round(tot/1e6, 2), img_per_sec=round(ips, 1),
                est_epoch_sec=round(est, 1), fits=est <= EPOCH_BUDGET)


rows = []
for nm, fz in [('simplecnn', False), ('cnn_bn_drop', False), ('myresnet18', False),
               ('resnet18', False), ('efficientnet_b0', False),
               ('efficientnet_v2_s', True), ('efficientnet_v2_s', False)]:
    try:
        r = speed_probe(nm, freeze=fz); r['freeze'] = fz; rows.append(r)
    except RuntimeError as e:
        print(f'{nm} 실패(메모리?): {str(e)[:70]}')
probe = pd.DataFrame(rows)
probe['이름'] = probe['model'] + np.where(probe['freeze'], ' (고정)', '')
probe['판정'] = np.where(probe['fits'], '○ 3분 이내', '✗ 초과')
display(probe[['이름', 'params_M', 'img_per_sec', 'est_epoch_sec', '판정']])

plt.figure(figsize=(9, 3.5))
plt.bar(probe['이름'], probe['est_epoch_sec'],
        color=['tab:green' if f else 'tab:red' for f in probe['fits']])
plt.axhline(EPOCH_BUDGET, ls='--', c='k'); plt.text(0, EPOCH_BUDGET*1.03, '3분 예산', fontsize=9)
plt.ylabel('1 에폭 예상 시간(초)'); plt.title('모델별 에폭 시간 — 과제 조건 대비')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

### 3분을 못 맞출 때의 대응 순서

1. **AMP(혼합정밀)** — GPU면 `train_model` 에서 이미 켜져 있다. 가장 싸다
2. **입력 해상도를 낮춘다** (`IMAGE_SIZE` 224 → 160). 연산량이 약 절반
3. **더 가벼운 백본** / **백본 고정**
4. **배치 크기 조정**, `NUM_WORKERS` 를 올려 데이터 로딩 병목 제거
5. (최후) 학습셋 일부만 사용 — 성능이 떨어지므로 마지막 수단

**초록색만 후보다.** 빨간색은 성능이 아무리 좋아도 이번 과제에서는 쓸 수 없다.

---
# §9. 실험 1 — 베이스라인  🔁 *여기서부터 에폭을 돌린다*

**기준점을 만든다.** 나중에 "macro-F1 0.9" 를 얻었을 때 그게 잘한 건지 문제가 쉬운 건지 구분할 자가 필요하다.

## 9-1. M0 — 딥러닝 없이 어디까지 가나 (`pro4_1` 의 아이디어)

혈액학자가 실제로 보는 것을 숫자로 만든다. **딥러닝의 추가 가치를 정직하게 재는 자**다.

| 특징 | 임상적 의미 |
|---|---|
| 핵 면적 | 세포 크기 |
| N/C 비 (핵/세포 면적비) | **림프구**는 이 값이 매우 큼 |
| 핵 오목성 | **호중구**의 분엽핵일수록 큼 |
| 세포질 붉은 정도 | **호산구**의 과립 |
| 핵 채도·명도 | 염색 강도 |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def handcrafted_features(path):
    rgb = np.asarray(Image.open(path).convert('RGB'))
    box, mask = detect_wbc_box(rgb)
    if box is None:
        box = (0, 0, rgb.shape[1], rgb.shape[0])
    x0, y0, x1, y1 = box
    roi = rgb[y0:y1, x0:x1].astype(np.float32) / 255.
    m = mask[y0:y1, x0:x1]
    if m.sum() < 10:
        return np.zeros(6, np.float32)

    cell = ndimage.binary_dilation(m, np.ones((3, 3)), iterations=12)   # 핵을 부풀려 세포 근사
    nuc_area, cell_area = m.mean(), max(cell.mean(), 1e-6)
    filled = ndimage.binary_fill_holes(ndimage.binary_closing(m, np.ones((9, 9))))
    concavity = float(1 - m.sum() / max(filled.sum(), 1))               # 분엽핵일수록 커진다
    cyto = cell & ~m
    redness = float((roi[..., 0] - roi[..., 1])[cyto].mean()) if cyto.any() else 0.0
    mx, mn = roi.max(-1), roi.min(-1)
    sat = np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0)
    return np.array([nuc_area, nuc_area/cell_area, concavity, redness,
                     float(sat[m].mean()), float(roi[m].mean())], np.float32)


CACHE_M0 = f'{RESULT_DIR}/m0_features.npz'
if os.path.exists(CACHE_M0):
    z = np.load(CACHE_M0); F_fit, y_fit_m0, F_orig = z['F_fit'], z['y_fit'], z['F_orig']
    print('M0 특징 캐시 사용')
else:
    rs = np.random.RandomState(0)
    sub = rs.choice(idx_fit, min(2000, len(idx_fit)), replace=False)
    t0 = time.time()
    F_fit = np.stack([handcrafted_features(p) for p in aug_df['path'].values[sub]])
    y_fit_m0 = y_aug[sub]
    F_orig = np.stack([handcrafted_features(p) for p in orig_df['path'].values])
    np.savez(CACHE_M0, F_fit=F_fit, y_fit=y_fit_m0, F_orig=F_orig)
    print(f'M0 특징 계산 {time.time()-t0:.0f}초')

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
clf.fit(F_fit, y_fit_m0)
pred_m0 = clf.predict(F_orig)
f1_m0 = f1_score(y_orig, pred_m0, average='macro', zero_division=0)
acc_m0 = accuracy_score(y_orig, pred_m0)
print(f'\nM0 (수작업 특징 + 로지스틱회귀) → ② 원본 검증  macro-F1 {f1_m0:.4f} / 정확도 {acc_m0:.4f}')
print(classification_report(y_orig, pred_m0, target_names=[KOR[c] for c in CLASS_NAMES],
                            zero_division=0))

## 9-2. M1~M5 — 딥러닝 베이스라인

같은 증강(`flip`), 같은 입력(ROI), 같은 시드로 **모델만 바꾼다.**
조기종료와 체크포인트는 **② 원본 검증 macro-F1** 기준이다 (§3-3).

In [ ]:
BASE = [
    ('M1_SimpleCNN',      dict(model_name='simplecnn',   pretrained=False, lr=1e-3, dropout=0.0)),
    ('M2_CNN_BN_Drop',    dict(model_name='cnn_bn_drop', pretrained=False, lr=1e-3, dropout=0.25)),
    ('M3_MyResNet18',     dict(model_name='myresnet18',  pretrained=False, lr=1e-3, dropout=0.0)),
    ('M4_EffNetV2S_고정',  dict(model_name='efficientnet_v2_s', freeze=True,  lr=1e-3, dropout=0.3)),
    ('M5_EffNetV2S_미세조정', dict(model_name='efficientnet_v2_s', freeze=False, lr=1e-4, dropout=0.3)),
]
base_res = {}
for rid, kw in BASE:
    r = run_experiment(rid, preset='flip', use_roi=True, epochs=FULL_EPOCHS,
                       patience=5, subset=SUBSET, **kw)
    if r: base_res[rid] = r
    print('-' * 92)

In [ ]:
for rid in ['M1_SimpleCNN', 'M5_EffNetV2S_미세조정']:
    if rid in base_res: plot_history(base_res[rid]['history'], rid)

In [ ]:
t = runs_table()
order = [r for r, _ in BASE]
b = t[t.run_id.isin(order)].set_index('run_id').reindex(order).reset_index()
b.insert(1, 'M0_수작업특징', '')
display(b[['run_id', 'params_M', 'best_epoch', 'epoch_sec',
           'orig_accuracy', 'orig_macro_f1', 'rand_macro_f1']].round(4))
print(f'참고 — M0 수작업 특징 macro-F1 {f1_m0:.4f} / 무작위 찍기 정확도 {1/NUM_CLASSES:.2f}')

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].bar(b.run_id, b.orig_macro_f1, color='tab:blue')
ax[0].axhline(f1_m0, ls='--', c='crimson'); ax[0].text(-.45, f1_m0+.01, 'M0 수작업특징', color='crimson', fontsize=9)
ax[0].set_ylabel('② 원본검증 macro-F1'); ax[0].set_title('성능(정직한 시험지)'); ax[0].tick_params(axis='x', rotation=20)
w = 0.38; x = np.arange(len(b))
ax[1].bar(x-w/2, b.rand_macro_f1, w, label='① 무작위 검증', color='tab:red')
ax[1].bar(x+w/2, b.orig_macro_f1, w, label='② 원본 검증', color='tab:blue')
ax[1].set_xticks(x); ax[1].set_xticklabels(b.run_id, rotation=20, ha='right')
ax[1].set_title('시험지에 따라 점수가 이만큼 달라진다'); ax[1].legend()
ax[2].bar(b.run_id, b.epoch_sec, color='tab:green'); ax[2].axhline(EPOCH_BUDGET, ls='--', c='r')
ax[2].set_ylabel('에폭당 시간(초)'); ax[2].set_title('비용 (빨간선 = 3분)'); ax[2].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

### 읽는 법

1. **M0 vs M1~M5** — 딥러닝의 추가 가치. M0 를 못 넘으면 딥러닝을 쓸 이유가 없다.
2. **M1 vs M2** — BatchNorm·Dropout(수업 2-10)의 효과.
3. **M3 vs M5** — **구조는 같고 사전학습 유무만 다르다.** 이 차이가 전이학습의 순수 이득이다.
4. **M4 vs M5** — 고정 vs 미세조정. M4 가 크게 못 미치면
   "현미경 이미지는 ImageNet 특징으로 표현되지 않는다"는 뜻이고, 수업 5-3 의 원칙과 일치한다.
5. **가운데 그래프가 §3 의 핵심 증거다.** 같은 모델인데 시험지에 따라 점수가 다르다.
   ① 이 ② 보다 훨씬 높다면, ① 은 외운 것을 다시 물어본 시험이다.

> 이 차이들이 **통계적으로 유의한지**는 §21 의 McNemar 검정에서 확인한다.

---
# §10. 실험 2 — 에폭을 몇으로 잡을 것인가  🔁

"충분한 에폭"은 감이 아니라 **곡선으로** 정한다.
여기서는 **조기종료를 끄고** 끝까지 돌려 과적합이 시작되는 지점을 눈으로 확인한다.

> **📘 수업 2-9 / 5-4**: 학습 손실은 계속 내려가는데 **검증 손실이 오르기 시작하면 외우기 시작한 것**이다.

In [ ]:
set_seed(42)
d_fit, d_rand, d_orig, _ = build_datasets('flip', True, subset=SUBSET)
LONG_EPOCHS = max(FULL_EPOCHS, 12)
long_model = build_model('simplecnn', pretrained=False)
hist_long = train_model(long_model,
                        make_loader(d_fit, shuffle=True), make_loader(d_orig, BATCH_SIZE*2),
                        epochs=LONG_EPOCHS, lr=1e-3, patience=LONG_EPOCHS,   # 조기종료 사실상 끔
                        ckpt_path=f'{RESULT_DIR}/ckpt/epoch_study.pt', label='에폭탐색',
                        extra_loaders={'①무작위': make_loader(d_rand, BATCH_SIZE*2)})

In [ ]:
e = np.arange(1, len(hist_long['loss'])+1)
gap = np.array(hist_long['val_loss']) - np.array(hist_long['loss'])
best_e = hist_long['best_epoch']

fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
ax[0].plot(e, hist_long['loss'], marker='o', label='학습')
ax[0].plot(e, hist_long['val_loss'], marker='o', label='②원본검증')
ax[0].axvline(best_e, ls='--', c='g'); ax[0].set_title('손실 — 검증이 꺾이는 지점'); ax[0].legend()
ax[1].plot(e, gap, marker='o', color='tab:red'); ax[1].axhline(0, c='k', lw=.8)
ax[1].set_title('과적합 간격 (검증 − 학습 손실)')
ax[2].plot(e, hist_long['val_macro_f1'], marker='o', color='tab:blue', label='②원본검증')
if '①무작위_macro_f1' in hist_long:
    ax[2].plot(e, hist_long['①무작위_macro_f1'], marker='s', color='tab:red', label='①무작위검증')
ax[2].axvline(best_e, ls='--', c='g'); ax[2].set_title('macro-F1'); ax[2].legend()
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'② 원본검증 macro-F1 최고: epoch {best_e} / 총 {len(e)} 에폭')
print(f'에폭당 {hist_long["epoch_sec_mean"]:.0f}초 → 3분 예산 대비 {EPOCH_BUDGET/max(hist_long["epoch_sec_mean"],1e-9):.1f}배 여유')
print('결론: best_epoch 가 최대 에폭에 붙어 있으면 덜 학습된 것이니 FULL_EPOCHS 를 늘린다.')
print('      한참 앞에서 멈췄으면 그 값이 곧 "충분한 에폭"이고, patience 는 그 뒤 몇 에폭을 기다릴지다.')

---
# §11. 실험 3 — 입력(ROI)과 증강  🔁

여기서 **§3 의 진단을 직접 확인**하고, 동시에 **ROI 와 증강의 효과**를 판정한다.

**조건을 하나만 바꾼다** — 모델·학습률·에폭·시드를 고정하고 입력/증강만 바꾼다.

In [ ]:
REF_MODEL, REF_LR = 'efficientnet_v2_s', 1e-4

# (1) ROI 효과: 전체 이미지 vs ROI crop
for use_roi in [False, True]:
    run_experiment(f'ROI_{int(use_roi)}', model_name=REF_MODEL, preset='flip', use_roi=use_roi,
                   lr=REF_LR, epochs=SCREEN_EPOCHS, patience=3, seed=42, subset=SUBSET)
    print('-' * 92)

t = runs_table(); roi_t = t[t.run_id.str.startswith('ROI_')].sort_values('use_roi')
display(roi_t[['run_id', 'use_roi', 'orig_macro_f1', 'orig_accuracy',
               'rand_macro_f1', 'epoch_sec']].round(4))
USE_ROI = bool(roi_t.sort_values('orig_macro_f1', ascending=False).iloc[0].use_roi)
print('선정된 입력:', 'ROI crop' if USE_ROI else '전체 이미지')

In [ ]:
# (2) 증강 프리셋 비교
for p in AUG_PRESETS:
    run_experiment(f'AUG_{p}', model_name=REF_MODEL, preset=p, use_roi=USE_ROI,
                   lr=REF_LR, epochs=SCREEN_EPOCHS, patience=3, seed=42, subset=SUBSET)
    print('-' * 92)

t = runs_table(); a = t[t.run_id.str.startswith('AUG_')].copy()
a['증강'] = a.run_id.str[4:]
a = a.sort_values('orig_macro_f1', ascending=False)
display(a[['증강', 'orig_macro_f1', 'orig_accuracy', 'rand_macro_f1', 'best_epoch', 'epoch_sec']].round(4))

x = np.arange(len(a)); w = 0.38
plt.figure(figsize=(9, 3.8))
plt.bar(x-w/2, a.rand_macro_f1, w, label='① 무작위 검증 (부풀려짐)', color='tab:red')
plt.bar(x+w/2, a.orig_macro_f1, w, label='② 원본 검증 (선택 기준)', color='tab:blue')
plt.xticks(x, a['증강'], rotation=15); plt.ylabel('macro-F1')
plt.title('증강 효과 — 시험지에 따라 순위가 뒤집힌다'); plt.legend(); plt.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

BEST_PRESET = str(a.iloc[0]['증강'])
print('선정된 증강:', BEST_PRESET)
print('① 기준 1등 :', str(a.sort_values('rand_macro_f1', ascending=False).iloc[0]['증강']),
      ' ← 이 둘이 다르면 §3 의 주장이 실증된 것이다')

### ⭐ 이 표가 §3 의 실증이다

- **① 무작위 검증 기준 1등**과 **② 원본 검증 기준 1등**이 다르게 나오면,
  "부풀려진 시험지로 고르면 잘못된 설정을 고른다"가 데이터로 증명된 것이다.
- 보통 ① 은 **증강을 적게 준 설정**을 좋아한다. 형제 사진을 외우는 데는 증강이 방해가 되기 때문이다.
- ② 는 **증강을 충분히 준 설정**을 좋아한다. 처음 보는 사진에는 증강이 도움이 되기 때문이다.

**ROI 효과도 같이 읽는다.** ROI 가 좋다면, 모델이 배경 적혈구가 아니라 세포를 보게 만든 것이
일반화에 도움이 되었다는 뜻이다 (§18 CAM 에서 다시 확인한다).

---
# §12. 실험 4 — 모델 비교  🔁

§8 에서 3분 예산을 통과한 후보들을 **정해진 증강·입력**으로 다시 비교한다.

In [ ]:
CAND = [m for m in ['resnet18', 'resnet34', 'efficientnet_b0', 'efficientnet_v2_s']
        if bool(probe.loc[(probe.model == m) & (~probe.freeze), 'fits'].max())]
print('비교 대상:', CAND)
for m in CAND:
    run_experiment(f'CMP_{m}', model_name=m, preset=BEST_PRESET, use_roi=USE_ROI,
                   lr=REF_LR, epochs=SCREEN_EPOCHS, patience=3, seed=42, subset=SUBSET)
    print('-' * 92)

t = runs_table(); c = t[t.run_id.str.startswith('CMP_')].sort_values('orig_macro_f1', ascending=False)
display(c[['run_id', 'params_M', 'epoch_sec', 'best_epoch', 'orig_macro_f1',
           'orig_accuracy', 'rand_macro_f1']].round(4))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(c.epoch_sec, c.orig_macro_f1, s=70)
for _, r in c.iterrows():
    ax.annotate(r.run_id[4:], (r.epoch_sec, r.orig_macro_f1), fontsize=8,
                xytext=(5, 4), textcoords='offset points')
ax.axvline(EPOCH_BUDGET, ls='--', c='r'); ax.set_xlabel('에폭당 실측 시간(초)')
ax.set_ylabel('② 원본검증 macro-F1'); ax.set_title('성능 vs 비용 (빨간선 = 3분)'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

ok = c[c.epoch_sec <= EPOCH_BUDGET]
top = ok.orig_macro_f1.max()
near = ok[ok.orig_macro_f1 >= top - 0.005]          # 최고와 0.5%p 이내면 동급으로 본다
pick = near.sort_values('epoch_sec').iloc[0]
BEST_MODEL = pick.run_id[4:]; TOP_MODEL = c.iloc[0].run_id[4:]
print(f'최고 성능 : {TOP_MODEL} ({top:.4f})')
print(f'최종 선정 : {BEST_MODEL} ({pick.orig_macro_f1:.4f}, {pick.epoch_sec:.0f}s, {pick.params_M}M)')
print('  기준 1) 3분 예산 준수  2) 최고와 0.5%p 이내  3) 그중 가장 빠름')
print('  → 이 차이가 진짜 없는지는 §21 McNemar 검정으로 확인한다')

---
# §13. 실험 5 — 하이퍼파라미터  🔁

**한 번에 하나씩** 바꾼다. 모든 판단은 ② 원본 검증으로 한다.

In [ ]:
# (1) 학습률 — 수업 5-3: 미세조정은 작게
for lr in [3e-4, 1e-4, 5e-5, 2e-5]:
    run_experiment(f'LR_{lr:g}', model_name=BEST_MODEL, preset=BEST_PRESET, use_roi=USE_ROI,
                   lr=lr, epochs=SCREEN_EPOCHS, patience=3, seed=42, subset=SUBSET)
    print('-' * 92)
t = runs_table(); l = t[t.run_id.str.startswith('LR_')].sort_values('lr')
display(l[['run_id', 'lr', 'best_epoch', 'orig_macro_f1', 'orig_accuracy', 'orig_loss']].round(5))
plt.figure(figsize=(6, 3.4))
plt.semilogx(l.lr, l.orig_macro_f1, marker='o'); plt.xlabel('학습률')
plt.ylabel('② 원본검증 macro-F1'); plt.title('학습률'); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
BEST_LR = float(l.sort_values('orig_macro_f1', ascending=False).iloc[0].lr)
print('선정된 학습률:', BEST_LR, '  ※ 최적값이 구간 양 끝이면 구간을 넓혀 다시 돌린다')

In [ ]:
# (2) 스케줄러 3종 (수업 3-15) + 드롭아웃 + weight decay + 클래스 가중치
for sch in ['plateau', 'cosine', 'step']:
    run_experiment(f'SCH_{sch}', model_name=BEST_MODEL, preset=BEST_PRESET, use_roi=USE_ROI,
                   lr=BEST_LR, scheduler=sch, epochs=SCREEN_EPOCHS, patience=3, seed=42, subset=SUBSET)
t = runs_table(); s = t[t.run_id.str.startswith('SCH_')].sort_values('orig_macro_f1', ascending=False)
display(s[['run_id', 'scheduler', 'best_epoch', 'orig_macro_f1', 'orig_loss']].round(5))
BEST_SCH = str(s.iloc[0].scheduler); print('선정된 스케줄러:', BEST_SCH)

grid = [dict(dropout=0.0, weight_decay=0.0,  use_class_weight=False),
        dict(dropout=0.3, weight_decay=0.0,  use_class_weight=False),
        dict(dropout=0.5, weight_decay=0.0,  use_class_weight=False),
        dict(dropout=0.3, weight_decay=1e-4, use_class_weight=False),
        dict(dropout=0.3, weight_decay=0.0,  use_class_weight=True)]
for i, g in enumerate(grid):
    run_experiment(f'REG_{i}', model_name=BEST_MODEL, preset=BEST_PRESET, use_roi=USE_ROI,
                   lr=BEST_LR, scheduler=BEST_SCH, epochs=SCREEN_EPOCHS, patience=3,
                   seed=42, subset=SUBSET, **g)
t = runs_table(); r = t[t.run_id.str.startswith('REG_')].sort_values('orig_macro_f1', ascending=False)
display(r[['run_id', 'dropout', 'weight_decay', 'class_weight',
           'orig_macro_f1', 'orig_accuracy', 'orig_loss']].round(5))
r0 = r.iloc[0]
BEST_REG = dict(dropout=float(r0.dropout), weight_decay=float(r0.weight_decay),
                use_class_weight=bool(int(r0.class_weight)))
print('선정된 정규화:', BEST_REG)

> **클래스 가중치에 대하여** — 학습에 쓰는 증강본은 클래스가 균형이라 가중치가 필요 없어 보인다.
> 그런데 **선택 기준인 ② 원본 검증은 심하게 불균형**(호중구 210 : 단핵구 20)하다.
> 소수 클래스 재현율을 끌어올리는 것이 macro-F1 에 유리할 수 있으므로 **실험으로 판정**했다.
> (수업 4-7 의 `pos_weight` 를 다중 클래스로 옮긴 것)

---
# §14. 실험 6 — 시드를 바꿔 반복  🔁

지금까지의 비교는 **시드 하나짜리 한 번씩**이라 차이가 설정 때문인지 우연인지 알 수 없다.
비교하려는 두 조건을 **여러 시드로 돌려** 점수쌍을 모은다. **같은 시드끼리 짝지어야** 다른 변동이 상쇄된다.

> **H0**: 두 증강 조건의 ② 원본 검증 macro-F1 평균이 같다 / **H1**: 다르다 (양측)
> → §21 의 가설검정 2 에서 판정

In [ ]:
AB = {f'A_{BEST_PRESET}': BEST_PRESET, 'B_none': 'none'}
if BEST_PRESET == 'none':
    AB = {'A_flip': 'flip', 'B_none': 'none'}
print(f'총 {len(SEEDS)*len(AB)} 회 학습 예정')
for tag, preset in AB.items():
    for sd in SEEDS:
        run_experiment(f'SEED_{tag}_s{sd}', model_name=BEST_MODEL, preset=preset, use_roi=USE_ROI,
                       lr=BEST_LR, scheduler=BEST_SCH, epochs=max(SCREEN_EPOCHS, 3),
                       patience=3, seed=sd, subset=SUBSET, verbose=False, **BEST_REG)
        print(f'  {tag} seed={sd} 완료')

t = runs_table(); sd_t = t[t.run_id.str.startswith('SEED_')].copy()
sd_t['조건'] = np.where(sd_t.preset == 'none', 'B_증강없음', 'A_최적증강')
piv = sd_t.pivot_table(index='seed', columns='조건', values='orig_macro_f1')
display(piv.round(4)); display(piv.describe().round(4).loc[['mean', 'std']])
plt.figure(figsize=(5.4, 3.6))
for _, rr in piv.iterrows():
    plt.plot(['A_최적증강', 'B_증강없음'], [rr['A_최적증강'], rr['B_증강없음']],
             marker='o', color='gray', alpha=.7)
plt.boxplot([piv['A_최적증강'], piv['B_증강없음']], positions=[0, 1], widths=.35)
plt.ylabel('② 원본검증 macro-F1'); plt.title('시드 반복 (같은 시드끼리 연결)')
plt.grid(alpha=.3); plt.show()

---
# §15. 최종 학습  🔁

지금까지 고른 설정으로 **전체 학습 데이터**를 써서 에폭을 충분히 주고 학습한다.

In [ ]:
FINAL_CFG = dict(model_name=BEST_MODEL, preset=BEST_PRESET, use_roi=USE_ROI,
                 lr=BEST_LR, scheduler=BEST_SCH, **BEST_REG)
print('최종 설정'); print(json.dumps(FINAL_CFG, ensure_ascii=False, indent=1))

final = run_experiment('FINAL', epochs=FULL_EPOCHS, patience=6, seed=42,
                       subset=None,                    # ← 전체 학습 데이터
                       **FINAL_CFG)
if final: plot_history(final['history'], '최종 모델')

t = runs_table(); fr = t[t.run_id == 'FINAL'].iloc[0]
print(f"best_epoch {int(fr.best_epoch)} / 최대 {FULL_EPOCHS} | 에폭당 {fr.epoch_sec:.0f}초"
      + ('  ⚠ 예산 초과' if fr.epoch_sec > EPOCH_BUDGET else '  ○ 3분 조건 충족'))
print(f"② 원본검증 macro-F1 {fr.orig_macro_f1:.4f} / ① 무작위검증 {fr.rand_macro_f1:.4f}")

---
# §16. 최종 평가 — 공식 TEST, 딱 한 번 ⭐

**여기서 처음으로 `dataset2-master/images/TEST` 폴더를 연다.**
전처리·모델·학습률·스케줄러·정규화·에폭 — 모든 선택이 끝난 뒤다.

> **📘 수업 5-4 의 사고**: 5장에서 `test_set` 경로에 `'train'` 이 들어가 "테스트 정확도"가
> 사실은 학습 데이터 점수였다. **분모를 반드시 확인한다.**

이 결과를 본 뒤에는 **모델도 파라미터도 바꾸지 않는다.**

In [ ]:
def load_run_model(run_id):
    t = runs_table(); row = t[t.run_id == run_id].iloc[0]
    m = build_model(str(row['model']), pretrained=False,      # 어차피 덮어쓴다 (수업 5-4)
                    freeze=False, dropout=float(row.get('dropout', 0) or 0),
                    image_size=IMAGE_SIZE)
    m.load_state_dict(torch.load(f'{RESULT_DIR}/ckpt/{run_id}.pt', map_location=device))
    return m.to(device).eval(), row

final_model, frow = load_run_model('FINAL')
_, _, ds_orig_f, ds_test_f = build_datasets(BEST_PRESET, USE_ROI)
L_test = make_loader(ds_test_f, BATCH_SIZE*2)
L_orig = make_loader(ds_orig_f, BATCH_SIZE*2)

m_test = evaluate(final_model, L_test, nn.CrossEntropyLoss())
m_orig = evaluate(final_model, L_orig, nn.CrossEntropyLoss())
np.savez_compressed(f'{RESULT_DIR}/preds/FINAL_TEST.npz',
                    test_probs=m_test['probs'], test_trues=m_test['trues'],
                    test_logits=m_test['logits'])

print(f'분모 확인: TEST {len(m_test["trues"])}장 (TEST 폴더 장수와 같아야 한다)')
print(summarize(m_test, '③ 공식 TEST'))
print(summarize(m_orig, '② 원본 검증'))

In [ ]:
def bootstrap_ci(y_true, y_pred, metric='accuracy', n_boot=2000, alpha=0.05, seed=42):
    """테스트셋을 복원추출로 다시 뽑아 반복 측정 → 추정값이 흔들리는 폭"""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    fn = {'accuracy': accuracy_score,
          'macro_f1': lambda a, b: f1_score(a, b, average='macro', zero_division=0),
          'balanced': balanced_accuracy_score}[metric]
    rs = np.random.RandomState(seed); n = len(y_true); v = []
    for _ in range(n_boot):
        i = rs.randint(0, n, n)
        if len(np.unique(y_true[i])) < 2: continue
        v.append(fn(y_true[i], y_pred[i]))
    v = np.array(v)
    return dict(point=float(fn(y_true, y_pred)),
                lo=float(np.percentile(v, 100*alpha/2)), hi=float(np.percentile(v, 100*(1-alpha/2))))

print('③ 공식 TEST — 95% 신뢰구간')
ci_test = {}
for met in ['accuracy', 'macro_f1', 'balanced']:
    ci = bootstrap_ci(m_test['trues'], m_test['preds'], met); ci_test[met] = ci
    print(f'  {met:9s} {ci["point"]:.4f}  [{ci["lo"]:.4f}, {ci["hi"]:.4f}]')

cm = plot_confusion(m_test['trues'], m_test['preds'], title='③ 공식 TEST — 혼동행렬(개수)')
plot_confusion(m_test['trues'], m_test['preds'], normalize=True,
               title='③ 공식 TEST — 행 정규화(= 클래스별 재현율)')
display(pd.DataFrame(classification_report(m_test['trues'], m_test['preds'],
        target_names=[KOR[c] for c in CLASS_NAMES], output_dict=True, zero_division=0)).T.round(4))

pairs = sorted([(KOR[CLASS_NAMES[i]], KOR[CLASS_NAMES[j]], int(cm[i, j]))
                for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j], key=lambda z: -z[2])
print('가장 많이 혼동한 쌍 (정답 → 예측)')
for a_, b_, n_ in pairs[:5]: print(f'  {a_} → {b_} : {n_}장')

In [ ]:
# 틀린 사례와 확신도 (수업 3-10 오분류 들여다보기)
wrong = np.where(m_test['preds'] != m_test['trues'])[0]
print(f'틀린 것 {len(wrong)}장 / {len(m_test["trues"])}장')
if len(wrong):
    sel = np.random.RandomState(0).choice(wrong, min(8, len(wrong)), replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(13, 6)); axes = axes.ravel()
    for ax_, i in zip(axes, sel):
        x, y = ds_test_f[int(i)]; p = m_test['preds'][i]; cf = m_test['probs'][i][p]
        ax_.imshow(denorm(x)); ax_.axis('off')
        ax_.set_title(f'정답 {KOR[CLASS_NAMES[y]]} / 예측 {KOR[CLASS_NAMES[p]]} ({cf:.2f})', fontsize=10)
    for ax_ in axes[len(sel):]: ax_.axis('off')
    plt.tight_layout(); plt.show()

conf = m_test['probs'].max(1); ok = m_test['preds'] == m_test['trues']
plt.figure(figsize=(6.4, 3.4))
plt.hist(conf[ok], bins=30, alpha=.6, label='맞힌 경우')
plt.hist(conf[~ok], bins=30, alpha=.6, label='틀린 경우')
plt.xlabel('예측 확신도 (softmax 최댓값)'); plt.ylabel('장수'); plt.legend(); plt.grid(alpha=.3)
plt.title('확신도 분포'); plt.show()
print(f'맞힘 평균 확신도 {conf[ok].mean():.3f} / 틀림 {conf[~ok].mean():.3f}')

---
# §17. 무누수 대조 트랙 — 원본 354장 5겹 교차검증  🔁

§16 까지는 증강본으로 학습했다. 증강본은 **누수 위험이 완전히 0 은 아니다**(§3).

그래서 **누수가 원천적으로 불가능한 실험**을 따로 돌린다.
원본 354장만 쓰면 **한 장이 곧 하나의 원본**이므로, 나누는 순간 형제가 생길 수 없다.
`3조`와 `pro4_1` 이 택한 길이 이것이다.

- **StratifiedKFold** 5겹: 클래스 비율을 유지하며 5번 나눠 학습·평가 (`pro4_1`)
- 단핵구가 20장뿐이라 **한 겹당 4장**이다. 단일 분할의 점수 차이는 대부분 운이므로
  **평균 ± 표준편차**로 보고한다. 표준편차가 겹치면 "차이가 있다"고 말하면 안 된다.
- 클래스 가중치를 쓴다 (수업 4-7 의 다중 클래스판)

이 트랙의 숫자가 **가장 보수적이고 정직한 성능 추정치**다.

In [ ]:
def run_cv_originals(model_name='efficientnet_v2_s', preset='flip', n_folds=CV_FOLDS,
                     epochs=CV_EPOCHS, lr=1e-4, use_class_weight=True, seed=42, tag='CV'):
    Xo = X_orig_roi if PRELOAD else None
    op = orig_df['path'].values
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    scores, recalls, all_true, all_pred = [], [], [], []

    for k, (tr_i, va_i) in enumerate(skf.split(np.arange(len(y_orig)), y_orig), 1):
        set_seed(seed + k)
        d_tr = WBCDataset(Xo[tr_i] if PRELOAD else None, y_orig[tr_i],
                          build_train_tf(preset), op[tr_i], True)
        d_va = WBCDataset(Xo[va_i] if PRELOAD else None, y_orig[va_i],
                          EVAL_TF, op[va_i], True)
        model = build_model(model_name, pretrained=True, freeze=False, dropout=0.3,
                            image_size=IMAGE_SIZE)
        cw = class_weights_from(y_orig[tr_i]) if use_class_weight else None
        h = train_model(model, make_loader(d_tr, shuffle=True), make_loader(d_va, BATCH_SIZE*2),
                        epochs=epochs, lr=lr, class_weight=cw, patience=max(3, epochs//3),
                        ckpt_path=f'{RESULT_DIR}/ckpt/{tag}_fold{k}.pt',
                        label=f'{tag}-fold{k}', verbose=False)
        model.load_state_dict(torch.load(h['ckpt_path'], map_location=device))
        m = evaluate(model, make_loader(d_va, BATCH_SIZE*2))
        scores.append(m['macro_f1']); recalls.append(m['recall_per_class'])
        all_true.append(m['trues']); all_pred.append(m['preds'])
        print(f'  fold {k}/{n_folds}  macro-F1 {m["macro_f1"]:.4f}  정확도 {m["accuracy"]:.4f}')
        del model
        if device.type == 'cuda': torch.cuda.empty_cache()

    s = np.array(scores)
    print(f'[{tag}] macro-F1 {s.mean():.4f} ± {s.std():.4f}')
    return dict(scores=s, recall=np.array(recalls).mean(0),
                trues=np.concatenate(all_true), preds=np.concatenate(all_pred))


print(f'=== 무누수 트랙: 원본 {len(y_orig)}장, {CV_FOLDS}겹 교차검증 ===')
cv = run_cv_originals(model_name=BEST_MODEL, preset=BEST_PRESET)

In [ ]:
print('클래스별 평균 재현율 (교차검증)')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {KOR[c]:4s} {cv["recall"][i]:.3f}   (표본 {int((y_orig==i).sum())}장)')
plot_confusion(cv['trues'], cv['preds'], normalize=True,
               title=f'무누수 트랙 — {CV_FOLDS}겹 교차검증 전체 혼동행렬')
print(f'\n무누수 트랙 macro-F1 = {cv["scores"].mean():.4f} ± {cv["scores"].std():.4f}')
print(f'③ 공식 TEST macro-F1  = {m_test["macro_f1"]:.4f}')
print(f'M0 수작업 특징        = {f1_m0:.4f}')

---
# §18. Grad-CAM 과 집중배율 — "정말 세포를 보고 있는가"

**과제 필수 항목.** 정확도가 높아도 **배경 적혈구를 외운 모델**이면 쓸 수 없다.

두 가지를 모두 그린다.

- **CAM**: 우리 모델은 `특징맵 → GAP → Linear` 구조라 **근사 없이 정확히** 성립한다.
  $$M_c(h,w)=\sum_k w_{c,k}\,f_k(h,w)$$
- **Grad-CAM**: 기울기를 채널 가중치로 쓴다. 구조와 무관해서 **CAM 의 검산**이 된다.

> **📘 `pro4_1` 이 알려준 함정 두 가지**
> 1. `nn.ReLU(inplace=True)` 는 backward 후크와 충돌해 RuntimeError 가 난다 → `inplace=False` (§6 에 반영)
> 2. 백본을 고정한 모델은 파라미터가 전부 `requires_grad=False` 라
>    **입력에 `requires_grad_(True)` 를 주지 않으면 후크가 아예 호출되지 않는다**

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval(); self.acts = self.grads = None
        self.handles = [target_layer.register_forward_hook(self._fwd),
                        target_layer.register_full_backward_hook(self._bwd)]
    def _fwd(self, m, i, o): self.acts = o                 # detach 하지 않는다(역전파가 지나가야 함)
    def _bwd(self, m, gi, go): self.grads = go[0].detach()
    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        x = x.clone().requires_grad_(True)                 # ★ 고정 백본에서도 기울기가 흐르도록
        logits = self.model(x)
        if class_idx is None: class_idx = int(logits.argmax(1))
        logits[0, class_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)      # 공간 평균 = 채널 가중치
        h = torch.relu((w * self.acts.detach()).sum(1)).squeeze(0)
        return (h / (h.max() + 1e-8)).detach().cpu().numpy(), class_idx
    def remove(self):
        for h in self.handles: h.remove()

@torch.no_grad()
def cam_map(model, x, class_idx=None):
    """CAM — 헤드 가중치를 그대로 쓴다 (GAP+Linear 구조에서 정확히 성립)"""
    x = x.unsqueeze(0).to(device)
    f = model.forward_features(x)
    logits = model.head(model.pool(f).flatten(1))
    c = int(logits.argmax(1)) if class_idx is None else class_idx
    h = torch.relu((f[0] * model.head.weight[c][:, None, None]).sum(0))
    return (h / (h.max() + 1e-8)).cpu().numpy(), c, torch.softmax(logits, 1)[0].cpu().numpy()

def last_conv_layer(model):
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    return convs[-1]

def upsample(h, size):
    return F.interpolate(torch.tensor(h, dtype=torch.float32)[None, None], size=size,
                         mode='bilinear', align_corners=False)[0, 0].numpy()

def overlay(ax, x, heat, title='', alpha=0.45):
    img = denorm(x); ax.imshow(img)
    ax.imshow(upsample(heat, img.shape[:2]), cmap='jet', alpha=alpha)
    ax.set_title(title, fontsize=10); ax.axis('off')

In [ ]:
# 클래스별로 맞힌 예시 한 장씩 — CAM 과 Grad-CAM 을 나란히
correct = np.where(m_test['preds'] == m_test['trues'])[0]
rs = np.random.RandomState(0)
def pick(c):
    ci = np.where(m_test['trues'] == c)[0]; okk = np.intersect1d(correct, ci)
    return int(rs.choice(okk if len(okk) else ci))
idx4 = [pick(c) for c in range(NUM_CLASSES)]

gc = GradCAM(final_model, last_conv_layer(final_model))
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(4*NUM_CLASSES, 7.2))
cors = []
for k, i in enumerate(idx4):
    x, y = ds_test_f[i]
    h1, p1, pr = cam_map(final_model, x)
    h2, _ = gc(x.unsqueeze(0).to(device))
    overlay(axes[0, k], x, h1, f'CAM · 정답 {KOR[CLASS_NAMES[y]]} / 예측 {KOR[CLASS_NAMES[p1]]} ({pr[p1]:.2f})')
    overlay(axes[1, k], x, h2, 'Grad-CAM (같은 곳이어야 정상)')
    if h1.shape == h2.shape: cors.append(np.corrcoef(h1.ravel(), h2.ravel())[0, 1])
plt.tight_layout(); plt.show()
if cors: print(f'CAM 과 Grad-CAM 의 상관계수 평균 {np.nanmean(cors):.3f} (1에 가까울수록 일치)')

In [ ]:
# 틀린 사례의 CAM — 여기가 가장 배울 게 많다
if len(wrong):
    wsel = rs.choice(wrong, min(8, len(wrong)), replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(14, 6.5)); axes = axes.ravel()
    for ax_, i in zip(axes, wsel):
        x, y = ds_test_f[int(i)]; h, p, pr = cam_map(final_model, x)
        overlay(ax_, x, h, f'정답 {KOR[CLASS_NAMES[y]]} / 예측 {KOR[CLASS_NAMES[p]]} ({pr[p]:.2f})')
    for ax_ in axes[len(wsel):]: ax_.axis('off')
    plt.suptitle('틀린 사례 — 모델이 본 곳'); plt.tight_layout(); plt.show()

## 집중배율 — 숫자로 채점한다 (`pro4_1` 의 지표)

눈으로 "세포를 보는 것 같다"는 **주관적**이다. 측정한다.

| 지표 | 정의 | 우연 수준 |
|---|---|---|
| **집중배율** ★ | (마스크 안 CAM 에너지 비율) ÷ (마스크 면적 비율) | **정확히 1.0** |
| 적중률 | CAM 최댓값 지점이 세포 안에 있는가 | 마스크 면적 비율 |

**주지표는 집중배율이다.** 우연 수준이 1.0 이라 "우연보다 몇 배 집중했는가"로 바로 읽힌다.
IoU 만 쓰면 안 된다 — 무작위 히트맵도 넓게 퍼지면 IoU 가 꽤 나온다.

> **읽을 때 주의**: 입력이 **이미 ROI crop 이면 세포가 화면을 거의 채우므로 집중배율의 천장이 낮다.**
> `pro4_1` 에서도 전체 이미지 입력은 5.37배, ROI 입력은 1.81배였다.
> **같은 입력 방식끼리만 비교**해야 하고, ROI 입력에서는 1.5배만 넘어도 의미가 있다.
> 그래서 여기서는 마스크를 세포 전체가 아니라 **핵 주변**으로 좁혀 잰다.

In [ ]:
def cell_mask_for(path, iters=4):
    """핵 중심 마스크를 ROI 좌표계·IMAGE_SIZE 로 맞춰 돌려준다.
    입력이 이미 ROI crop 이면 세포가 화면을 거의 채우므로, 팽창을 적게 줘서
    '핵 주변'을 기준으로 삼아야 집중배율이 의미를 가진다."""
    rgb = np.asarray(Image.open(path).convert('RGB'))
    box, mask = detect_wbc_box(rgb)
    if box is None: box = (0, 0, rgb.shape[1], rgb.shape[0])
    x0, y0, x1, y1 = BOX.get(path, box)
    m = ndimage.binary_dilation(mask[y0:y1, x0:x1], np.ones((3, 3)), iterations=iters)
    m = np.asarray(Image.fromarray((m*255).astype(np.uint8))
                   .resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)) > 127
    return m

def focus_report(model, dataset, paths, n=150, seed=0):
    sel = np.random.RandomState(seed).choice(len(paths), min(n, len(paths)), replace=False)
    energies, areas, hits = [], [], []
    for i in sel:
        m = cell_mask_for(paths[i])
        if not m.any(): continue
        x, _ = dataset[int(i)]
        h, _, _ = cam_map(model, x)
        H = np.clip(upsample(h, m.shape), 0, None)
        if H.sum() <= 0: continue
        energies.append(float(H[m].sum() / H.sum())); areas.append(float(m.mean()))
        iy, ix = np.unravel_index(H.argmax(), H.shape); hits.append(bool(m[iy, ix]))
    e, a = np.array(energies), np.array(areas)
    return dict(energy=e, area=a, focus=float((e / np.maximum(a, 1e-6)).mean()),
                hit=float(np.mean(hits)))

test_paths = aug_df['path'].values[idx_test]
fr_ = focus_report(final_model, ds_test_f, test_paths, n=150)
print(f'표본 {len(fr_["energy"])}장 (③ 공식 TEST)')
print(f'  CAM 에너지 중 세포 영역 비율 : {fr_["energy"].mean():.4f}')
print(f'  세포 영역의 면적 비율        : {fr_["area"].mean():.4f}  (우연 수준)')
print(f'  ★ 집중배율                   : {fr_["focus"]:.2f} 배   (1.0 = 아무 데나 본 것)')
print(f'  적중률(최댓값이 세포 안)      : {fr_["hit"]*100:.1f}%')
print('\n판정 기준(pro4_1): 집중배율 1.5 미만이면 근거를 신뢰할 수 없다 →',
      '통과' if fr_['focus'] >= 1.5 else '미달')

plt.figure(figsize=(6.4, 3.4))
plt.hist(fr_['area'], bins=30, alpha=.6, label='세포 면적 비율 (우연)')
plt.hist(fr_['energy'], bins=30, alpha=.6, label='CAM 에너지 비율 (실제)')
plt.xlabel('비율'); plt.ylabel('장수'); plt.legend(); plt.grid(alpha=.3)
plt.title('CAM 이 세포에 얼마나 몰려 있는가'); plt.show()

> **한계를 분명히 한다**: 이 마스크는 색으로 만든 근사치이지 의학적 분할이 아니다.
> 결론은 **"모델이 배경보다 세포 영역을 유의하게 더 본다"** 까지이고,
> "핵의 분엽 수를 세고 있다"까지는 말하지 않는다.

---
# §19. 염색 스트레스 테스트 (`pro4_1`)

검사실마다 염색 시약·시간·현미경 조명이 다르다.
**시험할 때만** 색조와 채도를 흔들어 성능이 얼마나 무너지는지 본다.
이 하락폭이 **"다른 병원에 그대로 가져갈 수 있는가"** 의 대리 지표다.

In [ ]:
def shift_stain(arr_u8, hue=0.0, sat=1.0):
    hsv = np.asarray(Image.fromarray(arr_u8).convert('HSV')).astype(np.float32)
    hsv[..., 0] = (hsv[..., 0] + hue * 255) % 255
    hsv[..., 1] = np.clip(hsv[..., 1] * sat, 0, 255)
    return np.asarray(Image.fromarray(hsv.astype(np.uint8), 'HSV').convert('RGB'))

base_arr = (X_aug_roi if USE_ROI else X_aug_full)[idx_test] if PRELOAD else None
rows = [{'조건': '원본', 'macro_F1': m_test['macro_f1'], '하락': 0.0}]
for hue, sat, tag in [(0.03, 1.0, '색조 +3%'), (-0.03, 1.0, '색조 −3%'),
                      (0.0, 0.8, '채도 ×0.8'), (0.0, 1.2, '채도 ×1.2'),
                      (0.05, 0.85, '색조+채도 동시')]:
    if PRELOAD:
        Xs = np.stack([shift_stain(a, hue, sat) for a in base_arr])
        ds = WBCDataset(Xs, y_aug[idx_test], EVAL_TF)
    else:
        continue
    f1s = evaluate(final_model, make_loader(ds, BATCH_SIZE*2))['macro_f1']
    rows.append({'조건': tag, 'macro_F1': f1s, '하락': f1s - m_test['macro_f1']})
df_stress = pd.DataFrame(rows); display(df_stress.round(4))
worst = float(df_stress['하락'].min())
print(f'최대 하락폭 {worst:+.4f}')
print('하락폭이 0.05 를 넘으면 타 검사실 이전 시 재학습·재검증이 필요하다는 근거가 된다.')

plt.figure(figsize=(7, 3.3))
plt.bar(df_stress['조건'], df_stress['macro_F1'], color=['#4C72B0'] + ['#DD8452']*(len(df_stress)-1))
plt.axhline(m_test['macro_f1'], color='gray', ls='--'); plt.ylabel('macro-F1')
plt.title('염색 색조 변화에 대한 강건성'); plt.xticks(rotation=15); plt.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

---
# §20. 보류 임계값 τ — 운영 곡선 (`pro4_1`)

현장에 쓴다면 **확신이 없는 건은 사람에게 넘기는 것**이 안전하다.
최대 확률(확신도)에 임계값 τ 를 걸어 τ 미만은 **"판독 보류 — 전문가 재확인"** 으로 넘긴다.

τ 가 의미를 가지려면 확률이 실제 맞을 확률과 비슷해야 하므로 **온도 스케일링**으로 먼저 보정한다.
온도 T 를 **② 원본 검증에서 학습**하고 ③ TEST 에 적용한다(테스트로 튜닝하지 않는다).

In [ ]:
def fit_temperature(logits, labels):
    lg = torch.tensor(logits, dtype=torch.float32); lb = torch.tensor(labels, dtype=torch.long)
    T = torch.ones(1, requires_grad=True)
    opt = torch.optim.LBFGS([T], lr=0.05, max_iter=80); nll = nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad(); loss = nll(lg / T.clamp(min=1e-2), lb); loss.backward(); return loss
    opt.step(closure); return float(T.detach().clamp(min=1e-2))

T = fit_temperature(m_orig['logits'], m_orig['trues'])     # ② 에서 학습
probs_cal = torch.softmax(torch.tensor(m_test['logits']) / T, dim=1).numpy()   # ③ 에 적용
print(f'온도 T = {T:.3f}  ( >1 이면 원래 모델이 과신하고 있었다는 뜻 )')

conf, pred = probs_cal.max(1), probs_cal.argmax(1)
rows = []
for t_ in np.arange(0.25, 1.00, 0.05):
    keep = conf >= t_
    if keep.sum() == 0: continue
    rows.append({'tau': round(float(t_), 2), '자동판독비율': float(keep.mean()),
                 '자동판독정확도': float((pred[keep] == m_test['trues'][keep]).mean()),
                 '보류건수': int((~keep).sum())})
if not rows:      # 확신도가 전부 낮아 어떤 τ 에서도 남는 게 없을 때(학습이 덜 된 경우)
    rows = [{'tau': 0.0, '자동판독비율': 1.0,
             '자동판독정확도': float((pred == m_test['trues']).mean()), '보류건수': 0}]
sweep = pd.DataFrame(rows); display(sweep.round(4))
TARGET = 0.95
ok_rows = sweep[sweep['자동판독정확도'] >= TARGET]
chosen = ok_rows.sort_values('자동판독비율', ascending=False).iloc[0] if len(ok_rows) else None

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(sweep['tau'], sweep['자동판독정확도'], 'o-', label='자동 판독 정확도')
ax[0].plot(sweep['tau'], sweep['자동판독비율'], 's-', label='자동 판독 비율')
ax[0].axhline(TARGET, color='crimson', ls='--', label=f'목표 {TARGET:.0%}')
if chosen is not None: ax[0].axvline(chosen['tau'], color='gray', ls=':')
ax[0].set_xlabel('보류 임계값 τ'); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)
ax[0].set_title('보류를 늘릴수록 자동 판독은 정확해진다')
ax[1].plot(sweep['자동판독비율'], sweep['자동판독정확도'], 'o-', color='#C44E52')
ax[1].axhline(TARGET, color='crimson', ls='--')
ax[1].set_xlabel('자동 판독 비율'); ax[1].set_ylabel('자동 판독 정확도')
ax[1].set_title('운영 곡선 — 정책 제언의 근거'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

if chosen is not None:
    print(f"선택된 τ = {chosen['tau']} → 자동판독 {chosen['자동판독비율']:.0%}, "
          f"그 정확도 {chosen['자동판독정확도']:.1%}, 보류 {int(chosen['보류건수'])}건은 전문가 재확인")
else:
    print(f'목표 정확도 {TARGET:.0%} 를 만족하는 τ 가 없다 → 현재 성능으로는 자동 판독을 권고할 수 없다')

---
# §21. 가설검정

**과제 필수 항목.** 중요한 건 "검정을 했다"가 아니라 **"이 상황에 왜 이 검정인가"** 를 설명하는 것이다.

### 검정을 고르는 기준

```
비교하려는 두 값이 같은 대상에서 나왔나?
 ├─ 예 (짝지은 자료) ─┬─ 이진 결과(맞음/틀림) → McNemar
 │                    └─ 연속값(점수)        → 대응표본 t검정 / 윌콕슨
 └─ 아니오 (독립 표본) ┬─ 비율 비교          → 두 비율 z검정
                       └─ 평균 비교          → 독립표본 t검정
```

**같은 테스트셋을 쓴 두 모델을 독립표본 검정으로 비교하는 것이 가장 흔한 실수다.**
같은 이미지들이라 두 결과가 강하게 연관돼 있고, 그 연관을 무시하면 검정력을 잃는다.

| # | 질문 | H0 | 검정 | 자료 |
|---|---|---|---|---|
| **1** | 전이학습이 밑바닥 CNN 보다 나은가 | 두 모델의 오분류율이 같다 | **McNemar** | ③ 공식 TEST (짝지음) |
| **2** | 증강이 효과가 있는가 | 두 조건의 평균 macro-F1 이 같다 | **대응표본 t + 윌콕슨** | 시드별 점수쌍 |
| **3** | **무작위 분할 검증이 부풀려졌는가** | ① 정확도 = ③ 정확도 | **두 비율 z검정** | 서로 다른 표본 |
| **4** | CAM 이 세포를 보는가 | 에너지 비율 = 면적 비율 | **대응표본 t (단측)** | 같은 이미지에서 잰 두 값 |

유의수준 α = 0.05, 검정이 4번이므로 마지막에 **Holm 보정**을 한다.

In [ ]:
ALPHA = 0.05

def mcnemar_test(y_true, pred_a, pred_b):
    a = (np.asarray(pred_a) == np.asarray(y_true)); b = (np.asarray(pred_b) == np.asarray(y_true))
    n01 = int((a & ~b).sum()); n10 = int((~a & b).sum()); n = n01 + n10
    if n == 0:            stat, p, meth = 0.0, 1.0, '불일치 없음'
    elif n < 25:          stat, p, meth = float(min(n01, n10)), float(stats.binomtest(min(n01, n10), n, 0.5).pvalue), '정확 이항검정'
    else:                 stat = (abs(n01-n10)-1)**2/n; p = float(stats.chi2.sf(stat, 1)); meth = '카이제곱(연속성 보정)'
    return dict(b=n01, c=n10, n_disc=n, statistic=float(stat), p_value=p, method=meth,
                acc_A=float(a.mean()), acc_B=float(b.mean()), diff=float(a.mean()-b.mean()),
                table=[[int((a&b).sum()), n01], [n10, int((~a&~b).sum())]])

# ── 검정 1 : 최종(전이학습) vs M1 밑바닥 CNN  — 같은 ③ TEST 에서
base_model, _ = load_run_model('M1_SimpleCNN')
m_test_base = evaluate(base_model, L_test, nn.CrossEntropyLoss())
r1 = mcnemar_test(m_test['trues'], m_test['preds'], m_test_base['preds'])
display(pd.DataFrame(r1['table'], index=['A(최종) 맞음', 'A 틀림'],
                     columns=['B(밑바닥) 맞음', 'B 틀림']))
print('=== 검정 1: McNemar ===')
print('  H0: 두 모델의 오분류율이 같다   H1: 다르다 (양측)')
print(f"  b={r1['b']} (A만 맞힘), c={r1['c']} (B만 맞힘), 불일치 {r1['n_disc']}")
print(f"  {r1['method']}  통계량 {r1['statistic']:.4f}  p = {r1['p_value']:.4e}")
print(f"  정확도 {r1['acc_A']:.4f} vs {r1['acc_B']:.4f}  차이 {r1['diff']:+.4f}")
print('  판정:', 'H0 기각 → 두 모델의 성능은 통계적으로 다르다' if r1['p_value'] < ALPHA
      else 'H0 기각 못함 → 차이를 입증하지 못했다')

In [ ]:
# (선택) 최고 성능 모델과 최종 선정 모델의 차이가 유의하지 않음을 보인다
if TOP_MODEL != BEST_MODEL and already_run(f'CMP_{TOP_MODEL}'):
    top_model, _ = load_run_model(f'CMP_{TOP_MODEL}')
    m_top = evaluate(top_model, L_test)
    rx = mcnemar_test(m_test['trues'], m_test['preds'], m_top['preds'])
    print(f"최종({BEST_MODEL}) vs 최고성능({TOP_MODEL}) : "
          f"{rx['acc_A']:.4f} vs {rx['acc_B']:.4f}, p = {rx['p_value']:.4f}")
    print('  →', '차이 없음 → 더 싼 모델을 쓴 것이 정당하다' if rx['p_value'] >= ALPHA
          else '차이 있음 → 성능이 더 중요하면 최고 모델을 재검토한다')
else:
    print('최고 성능 모델이 곧 최종 모델이라 추가 비교 생략')

In [ ]:
# ── 검정 2 : 증강 효과 (대응표본)
a_col = [c for c in piv.columns if c.startswith('A_')][0]
b_col = [c for c in piv.columns if c.startswith('B_')][0]
a_s, b_s = piv[a_col].values, piv[b_col].values
d = a_s - b_s
t2, p2 = stats.ttest_rel(a_s, b_s)
try:    w2, pw2 = stats.wilcoxon(a_s, b_s)
except ValueError: w2, pw2 = float('nan'), float('nan')
sd_ = d.std(ddof=1); ci2 = (d.mean()-1.96*sd_/np.sqrt(len(d)), d.mean()+1.96*sd_/np.sqrt(len(d)))
cohen = d.mean()/sd_ if sd_ > 0 else float('nan')

print('=== 검정 2: 대응표본 검정 (최적 증강 vs 증강 없음) ===')
print('  H0: 두 조건의 평균 macro-F1 이 같다   H1: 다르다 (양측)')
print(f'  n = {len(d)}쌍  평균 {a_s.mean():.4f} vs {b_s.mean():.4f}  차이 {d.mean():+.4f}')
print(f'  차이의 95% 신뢰구간 [{ci2[0]:+.4f}, {ci2[1]:+.4f}]')
print(f'  대응표본 t : t = {t2:.3f}, p = {p2:.4f}')
print(f'  윌콕슨     : W = {w2:.1f}, p = {pw2:.4f}   (정규성 가정이 약한 대안)')
print(f"  효과크기 Cohen's d = {cohen:.2f} ({'큼' if abs(cohen)>0.8 else '중간' if abs(cohen)>0.5 else '작음'})")
print('  판정:', 'H0 기각 → 증강이 성능에 영향을 준다' if p2 < ALPHA
      else 'H0 기각 못함 → 이 자료로는 차이를 입증하지 못했다')
print('\n  ※ "기각 못 함"은 "같다"가 아니다. 표본이 작으면 실제 차이도 못 잡는다.')
print('     → "차이가 없다"가 아니라 "이 자료로는 입증하지 못했다"로 쓴다.')

In [ ]:
# ── 검정 3 : ① 무작위 분할 검증이 부풀려졌는가  ⭐ 이 프로젝트의 핵심 검정
m_rand_final = evaluate(final_model, make_loader(build_datasets(BEST_PRESET, USE_ROI)[1], BATCH_SIZE*2))
k1 = int((m_rand_final['preds'] == m_rand_final['trues']).sum()); n1 = len(m_rand_final['trues'])
k2 = int((m_test['preds'] == m_test['trues']).sum());             n2 = len(m_test['trues'])
p1_, p2_ = k1/n1, k2/n2
pp = (k1+k2)/(n1+n2); se = math.sqrt(pp*(1-pp)*(1/n1 + 1/n2))
z3 = (p1_-p2_)/se if se > 0 else 0.0
p3 = float(2*stats.norm.sf(abs(z3)))
se_d = math.sqrt(p1_*(1-p1_)/n1 + p2_*(1-p2_)/n2)

print('=== 검정 3: 두 비율 z검정 — ① 무작위 검증 vs ③ 공식 TEST ===')
print('  H0: 두 시험지의 정확도가 같다   H1: 다르다 (양측)')
print(f'  ① 무작위 검증 : {k1}/{n1} = {p1_:.4f}   (증강본 TRAIN 에서 무작위로 뗀 20%)')
print(f'  ③ 공식 TEST   : {k2}/{n2} = {p2_:.4f}   (제작자가 따로 만든 시험지)')
print(f'  차이 {p1_-p2_:+.4f}  95% 신뢰구간 [{p1_-p2_-1.96*se_d:+.4f}, {p1_-p2_+1.96*se_d:+.4f}]')
print(f'  z = {z3:.3f}  p = {p3:.4e}')
print('  판정:', 'H0 기각 → 두 시험지의 난이도가 통계적으로 다르다' if p3 < ALPHA else 'H0 기각 못함')
print()
if p3 < ALPHA and p1_ > p2_:
    print('  ▶ 해석: ① 이 ③ 보다 유의하게 높다 = **무작위 분할 검증이 성능을 부풀린다**.')
    print('     같은 원본에서 나온 형제 사진이 학습과 검증에 함께 들어갔기 때문이다.')
    print('     이 노트북이 ② 원본 검증으로 모델을 고른 이유가 이것이고, 그 결정이 데이터로 정당화됐다.')
elif p3 >= ALPHA:
    print('  ▶ 해석: 두 시험지의 난이도 차이를 입증하지 못했다. 누수가 없다는 증거는 아니다.')

fig, ax = plt.subplots(figsize=(5.4, 3.2))
ci_r = bootstrap_ci(m_rand_final['trues'], m_rand_final['preds'], 'accuracy')
ax.errorbar([0, 1], [ci_r['point'], ci_test['accuracy']['point']],
            yerr=[[ci_r['point']-ci_r['lo'], ci_test['accuracy']['point']-ci_test['accuracy']['lo']],
                  [ci_r['hi']-ci_r['point'], ci_test['accuracy']['hi']-ci_test['accuracy']['point']]],
            fmt='o', capsize=6, ms=8)
ax.set_xticks([0, 1]); ax.set_xticklabels(['① 무작위 검증', '③ 공식 TEST'])
ax.set_ylabel('정확도'); ax.set_title('시험지에 따른 정확도와 95% 신뢰구간'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── 검정 4 : CAM 이 세포를 보는가 (단측 대응표본)
e_, a_ = fr_['energy'], fr_['area']
t4, p4 = stats.ttest_rel(e_, a_, alternative='greater')
dd = e_ - a_
print('=== 검정 4: 대응표본 t검정 (단측) ===')
print('  H0: CAM 에너지 비율 = 세포 면적 비율   H1: 에너지 비율이 더 크다')
print(f'  n = {len(dd)}장  에너지 {e_.mean():.4f} vs 면적 {a_.mean():.4f}  차이 {dd.mean():+.4f}')
print(f'  t = {t4:.3f}  p = {p4:.4e}   집중배율 {fr_["focus"]:.2f}배')
print(f"  효과크기 Cohen's d = {dd.mean()/dd.std(ddof=1):.2f}")
print('  판정:', 'H0 기각 → CAM 은 세포 영역에 유의하게 집중한다' if p4 < ALPHA
      else 'H0 기각 못함 → 근거가 세포에 있다고 말할 수 없다')

## 다중비교 보정 (Holm-Bonferroni)

검정을 4번 하면 실제로 아무 차이가 없어도 하나가 우연히 p<0.05 가 될 확률이
$1-0.95^4 \approx 18.5\%$ 로 올라간다. p값을 작은 것부터 정렬해 $k$번째에 $(m-k+1)$ 을 곱해 보정한다.

In [ ]:
def holm(pvals, names, alpha=0.05):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(run, 1.0)
    return pd.DataFrame({'가설': names, 'p (원본)': p, 'p (Holm 보정)': adj,
                         '판정': np.where(adj < alpha, '유의 (H0 기각)', '유의하지 않음')})

tests = [('1. 전이학습 vs 밑바닥CNN (McNemar)',      r1['p_value']),
         ('2. 증강 효과 (대응표본 t)',               float(p2)),
         ('3. ①무작위검증 vs ③공식TEST (두 비율)',   p3),
         ('4. CAM 이 세포를 보는가 (단측 t)',        float(p4))]
res_h = holm([p for _, p in tests], [n for n, _ in tests], ALPHA)
display(res_h.style.format({'p (원본)': '{:.3e}', 'p (Holm 보정)': '{:.3e}'}))

---
# §22. 최종 결론

In [ ]:
summary = pd.DataFrame([
    dict(구분='무작위로 찍기',                     정확도=1/NUM_CLASSES, macroF1=np.nan, 시험지='—'),
    dict(구분='M0 수작업 특징 + 로지스틱회귀',       정확도=acc_m0,        macroF1=f1_m0,  시험지='② 원본'),
    dict(구분='M1 밑바닥 CNN',                     정확도=m_test_base['accuracy'], macroF1=m_test_base['macro_f1'], 시험지='③ 공식 TEST'),
    dict(구분=f'최종 모델 ({BEST_MODEL})',          정확도=m_test['accuracy'],      macroF1=m_test['macro_f1'],      시험지='③ 공식 TEST'),
    dict(구분='최종 모델 — ② 원본 검증',            정확도=m_orig['accuracy'],      macroF1=m_orig['macro_f1'],      시험지='② 원본'),
    dict(구분='최종 모델 — ① 무작위 검증(부풀려짐)', 정확도=m_rand_final['accuracy'], macroF1=m_rand_final['macro_f1'], 시험지='① 무작위'),
    dict(구분=f'무누수 트랙 ({CV_FOLDS}겹 교차검증)', 정확도=np.nan,        macroF1=cv['scores'].mean(), 시험지='원본 CV'),
]).round(4)
display(summary)

t = runs_table()
print('=' * 78)
print('최종 모델 :', BEST_MODEL, '| 증강', BEST_PRESET, '| 입력', 'ROI' if USE_ROI else '전체',
      '|', f'{IMAGE_SIZE}px', '| lr', BEST_LR, '|', BEST_SCH, '|', BEST_REG)
print(f"③ 공식 TEST 정확도 {ci_test['accuracy']['point']:.4f}  "
      f"95% CI [{ci_test['accuracy']['lo']:.4f}, {ci_test['accuracy']['hi']:.4f}]")
print(f"③ 공식 TEST macro-F1 {ci_test['macro_f1']['point']:.4f}  "
      f"95% CI [{ci_test['macro_f1']['lo']:.4f}, {ci_test['macro_f1']['hi']:.4f}]")
print(f"무누수 트랙 macro-F1 {cv['scores'].mean():.4f} ± {cv['scores'].std():.4f}")
print(f"CAM 집중배율 {fr_['focus']:.2f}배 | 염색 스트레스 최대 하락 {worst:+.4f}")
print(f"에폭당 {float(t[t.run_id=='FINAL'].epoch_sec.iloc[0]):.0f}초 / 예산 {EPOCH_BUDGET}초 → "
      f"{'조건 충족' if float(t[t.run_id=='FINAL'].epoch_sec.iloc[0]) <= EPOCH_BUDGET else '조건 위반'}")
print(f'총 실험 {len(t)}건')
print('=' * 78)
t.to_csv(f'{RESULT_DIR}/전체실험기록.csv', index=False, encoding='utf-8-sig')

In [ ]:
show = ['run_id','model','preset','use_roi','lr','scheduler','dropout','class_weight','seed',
        'best_epoch','epochs_run','epoch_sec','orig_macro_f1','orig_accuracy','rand_macro_f1']
display(t[[c for c in show if c in t.columns]].sort_values('orig_macro_f1', ascending=False).round(4))

## 누수 방지 대조표 — 무엇을 어떻게 막았나

| 누수 가능성 | 이 노트북의 대응 | 근거 |
|---|---|---|
| 사전 증강본이 검증셋에 형제로 들어감 | **모델 선택을 ② 원본 검증으로** 했다 | §3-3, §11, 검정 3 |
| 검증/시험 데이터에 증강이 걸림 | 학습 변환과 평가 변환을 **분리**했다 | §5-2 (수업 5-2) |
| 테스트를 보고 모델·파라미터를 고름 | ③ 은 §16 에서 **딱 한 번** 열었고 그 뒤 아무것도 바꾸지 않았다 | §16 |
| 테스트로 임계값·온도를 맞춤 | 온도 T 와 τ 는 **② 에서 학습**해 ③ 에 적용했다 | §20 |
| 배경(적혈구 배열)을 외움 | **ROI crop** 으로 세포만 남기고, CAM 집중배율로 검증했다 | §4, §18, 검정 4 |
| 라벨을 추측해서 채움 | 다중 라벨·BASOPHIL 은 **제외**했다. 추측하지 않았다 | §2 |
| 원본 단위 분리가 안 됨 | 부모 매칭을 시도했으나 **신뢰 불가**로 판정 → 대신 **무누수 트랙**(§17)을 따로 돌려 대조했다 | §3-1, §17 |

## 세 노트북 대비 이 노트북이 개선한 점

| | 3조 | pro4_1 | 이전 전체과정 | **이 노트북** |
|---|---|---|---|---|
| 데이터 | 원본 354 | 원본 354 | 증강본 12,444 | **증강본 + 원본 둘 다** |
| 모델 선택 기준 | 원본 검증 | 교차검증 | ① 무작위 검증(부풀려짐) | **② 원본 검증** |
| 누수 진단 | 경로 중복 확인 | pHash (검증 안 함) | 픽셀 유사도 | **진단 도구를 먼저 검증**하고 행동으로 진단 |
| ROI | 없음 | 있음 | 없음 | **있음 + 화각 정규화 용도로 확장** |
| 클래스 가중치 | 없음 | 있음 | 없음(불필요 판단) | **실험으로 판정** |
| 통계 검정 | 안 함(배운 범위 밖) | 안 함 | 4종 | **4종 + Holm + 누수 검정 추가** |
| 운영 관점 | 없음 | τ, 염색 스트레스 | 없음 | **둘 다 포함** |

## 보고서 뼈대

1. **문제** — 말초혈액 도말에서 백혈구 4종 분류. BASOPHIL 은 원본 3장이라 제외 (§2)
2. **데이터의 함정** — 12,444장은 366장을 불린 것. 무작위 분할 검증은 성능을 부풀린다 (§3, 검정 3)
3. **전처리** — 색 기반 ROI 검출로 배경을 제거하고 원본·증강본의 화각을 맞춤 (§4)
4. **모델** — 3분 예산을 먼저 재고(§8), 밑바닥→BN/Dropout→잔차→고정→미세조정으로 이득을 분해(§9)
5. **선택** — 모든 결정은 ② 원본 검증. ③ 공식 TEST 는 마지막에 한 번 (§11~§16)
6. **성능** — ③ TEST 정확도 X (95% CI), macro-F1 Y. 무누수 트랙 Z ± s (§16, §17)
7. **근거** — CAM 집중배율 F배로 세포를 보고 있음을 정량 확인 (§18, 검정 4)
8. **운영** — 염색 색조 변화에 대한 강건성(§19), 보류 임계값 τ 운영 곡선(§20)
9. **검정** — 자료 구조에 맞는 검정 4종 + Holm 보정 (§21)
10. **한계** — ① 원본 단위 분리 미보장 ② BASOPHIL 미포함 ③ 단핵구 20장으로 표본 부족
    ④ 환자 ID 부재로 같은 환자 여부 확인 불가 ⑤ 단일 장비·단일 염색 조건

> **"이게 베스트다"라고 말할 때 반드시 함께 말할 것**
> 어떤 후보들과 비교했는지 · 어떤 기준으로 골랐는지 · 그 차이가 통계적으로 유의한지 ·
> 그리고 **어떤 시험지에서 잰 숫자인지**.